# Notebook 22 — Reinforcement Learning for Autonomous Flight

**Autonomous Flight Workshop** | Block 8: Learning-Based Control

---

## What You'll Learn

Classical controllers (PID, SE(3) geometric) require an accurate dynamics model and
careful hand-tuning. **Reinforcement learning (RL)** can *learn* control policies
directly from interaction, adapt to unmodeled dynamics, and generalize via
domain randomization. This notebook builds the full RL-for-flight pipeline from
first principles.

### Topics
1. **RL problem formulation** — MDP, state/action spaces, reward design
2. **Policy Gradient Theorem** — likelihood ratio trick, variance reduction
3. **Generalized Advantage Estimation (GAE)** — bias-variance trade-off
4. **PPO (Proximal Policy Optimization)** — clipped surrogate objective
5. **SAC (Soft Actor-Critic)** — maximum entropy RL, reparameterization trick
6. **Reward engineering** — dense vs sparse, potential-based shaping
7. **Navigation with obstacles** — depth-ray observations, collision avoidance
8. **PPO vs SAC vs classical control** — systematic comparison
9. **Domain randomization** — sim-to-real transfer
10. **Curriculum learning** — progressive difficulty

### Key References
- Schulman et al., "Proximal Policy Optimization Algorithms", *arXiv:1707.06347*, 2017
- Haarnoja et al., "Soft Actor-Critic: Off-Policy Maximum Entropy Deep RL with a Stochastic Actor", *ICML*, 2018
- Fujimoto et al., "Addressing Function Approximation Error in Actor-Critic Methods" (TD3), *ICML*, 2018
- Hwangbo et al., "Learning Agile and Dynamic Motor Skills for Legged Robots", *Science Robotics*, 2019
- Schulman et al., "High-Dimensional Continuous Control Using Generalized Advantage Estimation", *ICLR*, 2016
- Zhang et al., "Vision-Based Learning for Drones: A Survey", *IEEE TNNLS*, 2025
- Xu et al., "SimpleFlight: A Minimalist Framework for Zero-Shot Sim-to-Real of Quadrotors", 2025
- Bauersfeld et al., "RAPTOR: Foundation Policies for Quadrotor Control", *Science Robotics*, 2025
- "Comprehensive Review of RL for Autonomous Drone Systems", *Atlantis Press*, 2026

## Why Reinforcement Learning for Drone Control?

Before diving into algorithms, it is worth understanding *where* RL fits in
the broader quadrotor control stack — and why it is both promising and
difficult.

### The Control Hierarchy

Quadrotor flight is typically decomposed into nested control loops, each
operating at a different timescale:

| Level | Timescale | Input → Output | Classical Approach |
|:------|:----------|:---------------|:-------------------|
| **Low-level (attitude)** | 250–1000 Hz | Desired angles → motor PWM | PID on roll/pitch/yaw rates |
| **Mid-level (velocity)** | 50–100 Hz | Desired velocity → desired attitude | PID or geometric (SE(3)) controller |
| **High-level (planning)** | 1–10 Hz | Waypoints / task goals → velocity commands | A*, RRT, trajectory optimisation |

Classical controllers at each level require an **accurate dynamics model**
and **careful gain tuning**. When the model is wrong (wind gusts, payload
changes, worn motors), performance degrades.

### Where RL Fits

RL can replace or augment *any* level of this hierarchy:

- **Replacing low-level control**: RL outputs raw motor commands. Maximally
  flexible but requires billions of samples and careful sim-to-real transfer.
  This is the approach we implement in this notebook.
- **Replacing mid-level control**: RL outputs body-rate or thrust setpoints
  while a classical inner loop handles motor mixing. The **CTBR** (Collective
  Thrust + Body Rates) action space (SimpleFlight, 2025) is the current
  best practice for this paradigm.
- **Replacing high-level planning**: RL selects waypoints or behaviours
  while classical controllers execute them. This is the **indirect learning**
  paradigm (Zhang et al., *IEEE TNNLS*, 2025) and is the most deployment-ready.

### The Sample Efficiency Challenge

RL is notoriously **sample-hungry**. A state-of-the-art PPO policy for
quadrotor hover typically requires:

$$N_{\text{samples}} \approx 10^7 \text{–} 10^9 \text{ environment steps}$$

At a 50 Hz control rate, $10^8$ steps corresponds to $\sim 23$ days of
continuous real-world flight — clearly impractical. This is why **simulation
is essential**: GPU-parallelised simulators (Isaac Sim, Flightmare) can
generate $10^6$ steps/second, reducing wall-clock training time to minutes.

### The Sim-to-Real Gap: Deployment Bottleneck #1

Policies trained in simulation often fail on real hardware because the
simulator never perfectly captures:

- **Aerodynamic effects**: ground effect, blade-vortex interaction, turbulence
- **Actuator dynamics**: motor response time, ESC nonlinearities, battery voltage sag
- **Sensor noise**: IMU bias drift, vibration-induced noise, latency
- **Unmodelled dynamics**: flexible arms, asymmetric mass distribution

The 2026 Atlantis Press review ("Comprehensive Review of RL for Autonomous
Drone Systems") identifies **domain randomization** and **system identification**
as the two most effective strategies for closing this gap. We implement domain
randomization in Section 9.

> **Key takeaway**: RL is not a replacement for classical control — it is a
> *complement*. The most successful real-world drone systems use RL for
> high-level decision-making while retaining classical controllers for
> safety-critical inner loops.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from typing import Dict, List, Tuple
import time

from src.rl_agents import (
    ReplayBuffer, RolloutBuffer,
    MLPPolicy, MLPValueFunction,
    PPO, SAC,
    DroneHoverEnv, DroneNavigationEnv,
)
from src.drone import (
    QuadrotorParams, QuadrotorState, simulate_step, PIDController,
)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

np.random.seed(42)

## 1. RL Problem Formulation for Drone Control

We model quadrotor control as a **Markov Decision Process** (MDP)
$\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, r, \gamma)$:

### State space

$$\mathcal{S} \subseteq \mathbb{R}^{12}, \qquad
s = \begin{bmatrix}
p - p_d \\ v \\ \phi, \theta, \psi \\ \omega
\end{bmatrix}
= \begin{bmatrix}
\text{position error (3)} \\ \text{velocity (3)} \\ \text{Euler angles (3)} \\ \text{angular velocity (3)}
\end{bmatrix}$$

The position error $p - p_d$ makes the policy *goal-conditioned* — the
same policy works for any target position.

### Action space

$$\mathcal{A} = [-1, 1]^4$$

Four normalized motor commands, linearly mapped to thrust:
$f_i = \frac{a_i + 1}{2} \cdot f_{\max}$.

### Transition dynamics

$$s_{t+1} = f(s_t, a_t)$$

Given by the rigid-body quadrotor dynamics from Notebook 19 (Newton-Euler
equations integrated via RK4). The agent does *not* have access to this model —
it learns purely from sampled transitions.

### Reward function

A critical design choice. Dense, quadratic rewards provide smooth gradients:

$$r(s, a) = -w_p \|p - p_d\|^2 - w_v \|v\|^2 - w_\omega \|\omega\|^2 - w_a \|a\|^2 + r_{\text{alive}}$$

Sparse rewards ($+1$ only when $\|p - p_d\| < \epsilon$) are harder to learn but can
yield sharper final behavior.

### Objective

Find policy $\pi_\theta$ maximizing the expected discounted return:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} \gamma^t r(s_t, a_t)\right]$$

where $\gamma \in [0, 1)$ is the discount factor.

In [ ]:
# ── Explore the DroneHoverEnv state and action spaces ─────────────────

env = DroneHoverEnv(target_pos=np.array([0.0, 0.0, 1.0]), max_steps=200, dt=0.02, seed=42)

print(f"Observation dim: {env.obs_dim}")
print(f"Action dim:      {env.act_dim}")
print(f"Timestep:        {env.dt} s")
print(f"Max steps:       {env.max_steps}")
print(f"Target pos:      {env.target_pos}")
print()

# Run random episodes
n_episodes = 5
rng = np.random.RandomState(0)

episode_rewards = []
episode_lengths = []
all_trajectories = []
all_reward_curves = []

for ep in range(n_episodes):
    obs = env.reset()
    positions = [env.state.position.copy()]
    rewards = []
    done = False
    while not done:
        action = rng.uniform(-1, 1, size=4)
        obs, reward, done, info = env.step(action)
        positions.append(env.state.position.copy())
        rewards.append(reward)
    episode_rewards.append(sum(rewards))
    episode_lengths.append(len(rewards))
    all_trajectories.append(np.array(positions))
    all_reward_curves.append(np.array(rewards))

print("Random-policy episode statistics:")
print(f"  Mean return:   {np.mean(episode_rewards):.2f} ± {np.std(episode_rewards):.2f}")
print(f"  Mean length:   {np.mean(episode_lengths):.0f} ± {np.std(episode_lengths):.0f}")

# Plot
fig = plt.figure(figsize=(14, 5))

# 3D trajectories
ax1 = fig.add_subplot(131, projection='3d')
for i, traj in enumerate(all_trajectories):
    ax1.plot(traj[:, 0], traj[:, 1], traj[:, 2], alpha=0.7, label=f'Ep {i}')
ax1.scatter(*env.target_pos, color='red', s=100, marker='*', label='Target')
ax1.set_xlabel('X [m]'); ax1.set_ylabel('Y [m]'); ax1.set_zlabel('Z [m]')
ax1.set_title('Random-Policy Trajectories')
ax1.legend(fontsize=8)

# Reward per step
ax2 = fig.add_subplot(132)
for i, r_curve in enumerate(all_reward_curves):
    ax2.plot(r_curve, alpha=0.7, label=f'Ep {i}')
ax2.set_xlabel('Step'); ax2.set_ylabel('Reward')
ax2.set_title('Per-Step Rewards')
ax2.legend(fontsize=8)

# Position error over time
ax3 = fig.add_subplot(133)
for i, traj in enumerate(all_trajectories):
    errors = np.linalg.norm(traj - env.target_pos, axis=1)
    ax3.plot(errors, alpha=0.7, label=f'Ep {i}')
ax3.set_xlabel('Step'); ax3.set_ylabel('Position Error [m]')
ax3.set_title('Position Error')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Action space design insight**: The action $a \in [-1, 1]^4$ maps to motor thrust via $f_i = \tfrac{a_i + 1}{2} \cdot f_{\max}$, so action=0 produces 50% of maximum thrust. With our Crazyflie-like parameters ($m = 0.5$ kg, $f_{\max} \approx 68$ N per motor), the hover thrust is only $\sim 1.2$ N/motor — meaning the hover action is $a_{\text{hover}} \approx -0.96$. 

This is a deliberate design choice: the agent must discover that **nearly the entire [-1, 1] range is "too much thrust"**. This makes learning harder but more realistic — real quadrotors have significant thrust headroom for aggressive maneuvers. The SimpleFlight paper (2024) found that using **CTBR** (collective thrust + body rates) as the action space instead of raw motor commands improves sim-to-real transfer by 50%, precisely because CTBR normalizes around hover.

> **Practical tip**: If training is too slow, rescale the action space to center around hover: $f_i = f_{\text{hover}} + \Delta f_{\max} \cdot a_i$. This reduces the exploration problem from "find the right corner of action space" to "perturb around equilibrium".

## 2. Policy Gradient Theorem

All policy-gradient methods stem from a single result: we can estimate
the gradient of the expected return *without* differentiating through the
environment dynamics.

### Likelihood ratio trick

For any function $f(x)$ and parameterized distribution $p_\theta(x)$:

$$\nabla_\theta \mathbb{E}_{x \sim p_\theta}[f(x)]
= \nabla_\theta \int f(x) \, p_\theta(x) \, dx
= \int f(x) \, \nabla_\theta p_\theta(x) \, dx$$

Using the log-derivative identity $\nabla_\theta p_\theta = p_\theta \, \nabla_\theta \log p_\theta$:

$$= \int f(x) \, p_\theta(x) \, \nabla_\theta \log p_\theta(x) \, dx
= \mathbb{E}_{x \sim p_\theta}\left[f(x) \, \nabla_\theta \log p_\theta(x)\right]$$

### Application to RL

Let $\tau = (s_0, a_0, s_1, a_1, \ldots)$ be a trajectory with probability:

$$p_\theta(\tau) = p(s_0) \prod_{t=0}^{T} \pi_\theta(a_t | s_t) \, p(s_{t+1} | s_t, a_t)$$

Taking the log:

$$\log p_\theta(\tau) = \log p(s_0) + \sum_{t=0}^T \left[\log \pi_\theta(a_t|s_t) + \log p(s_{t+1}|s_t,a_t)\right]$$

The dynamics terms vanish under $\nabla_\theta$:

$$\nabla_\theta \log p_\theta(\tau) = \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t | s_t)$$

### Policy Gradient Theorem (Sutton et al., 1999)

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[\sum_{t=0}^T \nabla_\theta \log\pi_\theta(a_t|s_t) \, \hat{A}_t\right]}$$

where $\hat{A}_t$ is an *advantage estimate*. The simplest choice is
$\hat{A}_t = R_t = \sum_{t'=t}^T \gamma^{t'-t} r_{t'}$ (REINFORCE), but this
has **high variance**.

### Variance reduction via baselines

Subtracting a *baseline* $b(s_t)$ from the return does not change the
expectation of the gradient (since $\mathbb{E}[\nabla \log \pi \cdot b(s)] = 0$
for state-dependent baselines) but can drastically reduce variance. The
optimal baseline is close to $V^\pi(s_t)$, the value function:

$$\hat{A}_t = R_t - V^\pi(s_t)$$

This motivates learning a *critic* $V_\phi(s) \approx V^\pi(s)$ alongside
the policy — the **actor-critic** architecture.

## 3. Generalized Advantage Estimation (GAE)

GAE (Schulman et al., 2016) interpolates between low-bias/high-variance
Monte Carlo estimates and high-bias/low-variance one-step TD estimates.

### TD residual

The one-step temporal difference error is:

$$\delta_t^V = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This is an unbiased estimate of the advantage *if* the value function is
exact: $\mathbb{E}[\delta_t^V] = A^\pi(s_t, a_t)$ when $V = V^\pi$.

### Multi-step advantage estimates

Define the $k$-step advantage:

$$\hat{A}_t^{(k)} = \sum_{l=0}^{k-1} \gamma^l \delta_{t+l}^V
= -V(s_t) + r_t + \gamma r_{t+1} + \cdots + \gamma^{k-1} r_{t+k-1} + \gamma^k V(s_{t+k})$$

- $k = 1$: $\hat{A}_t^{(1)} = \delta_t^V$ — **one-step TD**, low variance, high bias
- $k = T - t$: $\hat{A}_t^{(T-t)} = R_t - V(s_t)$ — **Monte Carlo**, zero bias, high variance

### GAE: exponentially weighted average

GAE takes an exponentially-weighted average over all $k$-step estimates:

$$\hat{A}_t^{\text{GAE}(\gamma, \lambda)}
= (1 - \lambda) \sum_{k=1}^{\infty} \lambda^{k-1} \hat{A}_t^{(k)}
= \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}^V$$

**Proof** (telescoping):

$$\hat{A}_t^{\text{GAE}} = (1-\lambda)\left(\delta_t + \lambda(\delta_t + \gamma\delta_{t+1}) + \lambda^2(\delta_t + \gamma\delta_{t+1} + \gamma^2\delta_{t+2}) + \cdots\right)$$

Collecting terms for $\delta_{t+l}$, its coefficient is
$(1-\lambda) \gamma^l \sum_{j=0}^{\infty} \lambda^{l+j} = \gamma^l \lambda^l$.

### Special cases and the bias-variance trade-off

| $\lambda$ | Estimator | Bias | Variance |
|:---------:|:---------:|:----:|:--------:|
| 0 | One-step TD: $\hat{A}_t = \delta_t^V$ | High | Low |
| 1 | Monte Carlo: $\hat{A}_t = R_t - V(s_t)$ | Low | High |
| 0.95–0.99 | GAE (typical) | Moderate | Moderate |

**Why $\lambda$ controls bias-variance:**

- **Variance** comes from the randomness of future rewards. The $k$-step return
  $r_t + \gamma r_{t+1} + \cdots + \gamma^{k-1} r_{t+k-1}$ has variance proportional to $k$
  (sum of $k$ random variables). Since GAE weights the $k$-step estimate by $\lambda^{k-1}$,
  higher $\lambda$ gives more weight to longer (noisier) returns.

- **Bias** comes from the value function approximation error $\epsilon = V_\theta(s) - V^\pi(s)$.
  The $k$-step estimate bootstraps from $V(s_{t+k})$, contributing bias $\gamma^k \epsilon$.
  Lower $\lambda$ bootstraps sooner (small $k$), amplifying this bias. At $\lambda = 1$,
  we never bootstrap (pure MC), so bias is zero when $V$ is wrong.

Formally, for a value function with uniform error $\epsilon$:

$$
\text{Bias}[\hat{A}^{\text{GAE}}] = \frac{\gamma(1-\lambda)}{1-\gamma\lambda} \epsilon
$$

At $\lambda = 0$: bias $= \gamma\epsilon$. At $\lambda = 1$: bias $= 0$.

### Recursive computation

In practice, GAE is computed backward through the trajectory:

$$\hat{A}_T = \delta_T, \qquad \hat{A}_t = \delta_t + \gamma\lambda \, \hat{A}_{t+1}$$

In [ ]:
# ── Verify GAE computation numerically ──────────────────────────────────

gamma, lam = 0.99, 0.95
T = 20

rng = np.random.RandomState(123)
rewards = rng.randn(T) * 0.5 + 0.1
values  = rng.randn(T) * 0.3
dones   = np.zeros(T, dtype=bool)
last_value = rng.randn() * 0.3

# Method 1: Manual forward computation of GAE
deltas = np.zeros(T)
for t in range(T):
    next_val = values[t + 1] if t + 1 < T else last_value
    deltas[t] = rewards[t] + gamma * next_val - values[t]

gae_manual = np.zeros(T)
for t in range(T):
    gae_manual[t] = sum((gamma * lam)**l * deltas[t + l] for l in range(T - t))

# Method 2: Recursive backward (as in RolloutBuffer)
buf = RolloutBuffer()
for t in range(T):
    buf.add(
        obs=np.zeros(12),
        action=np.zeros(4),
        reward=rewards[t],
        done=dones[t],
        log_prob=0.0,
        value=values[t],
    )
buf.compute_returns_and_advantages(last_value, gamma, lam)
gae_buffer = np.array(buf.advantages)

max_diff = np.max(np.abs(gae_manual - gae_buffer))
print(f"Max |manual - buffer| = {max_diff:.2e}  (should be < 1e-12)")

# Method 3: Monte Carlo returns for comparison (lambda=1)
mc_returns = np.zeros(T)
G = last_value
for t in reversed(range(T)):
    G = rewards[t] + gamma * G
    mc_returns[t] = G
mc_advantage = mc_returns - values

# GAE with lambda=1 should equal MC advantage
buf2 = RolloutBuffer()
for t in range(T):
    buf2.add(np.zeros(12), np.zeros(4), rewards[t], dones[t], 0.0, values[t])
buf2.compute_returns_and_advantages(last_value, gamma, lam=1.0)
gae_lam1 = np.array(buf2.advantages)

print(f"Max |GAE(λ=1) - MC advantage| = {np.max(np.abs(gae_lam1 - mc_advantage)):.2e}")

# GAE with lambda=0 should equal one-step TD
buf3 = RolloutBuffer()
for t in range(T):
    buf3.add(np.zeros(12), np.zeros(4), rewards[t], dones[t], 0.0, values[t])
buf3.compute_returns_and_advantages(last_value, gamma, lam=0.0)
gae_lam0 = np.array(buf3.advantages)

print(f"Max |GAE(λ=0) - δ_t| = {np.max(np.abs(gae_lam0 - deltas)):.2e}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
returns_buf = np.array(buf.returns)
ax.plot(mc_returns, 'o-', label='MC Returns ($\\lambda=1$)', alpha=0.8)
ax.plot(returns_buf, 's-', label=f'GAE Returns ($\\lambda={lam}$)', alpha=0.8)
ax.plot(values, '^-', label='Value estimates $V(s_t)$', alpha=0.8)
ax.set_xlabel('Time step $t$')
ax.set_ylabel('Value')
ax.set_title('Returns and Value Estimates')
ax.legend()

ax = axes[1]
ax.plot(mc_advantage, 'o-', label='MC Advantage ($\\lambda=1$)', alpha=0.8)
ax.plot(gae_buffer, 's-', label=f'GAE ($\\lambda={lam}$)', alpha=0.8)
ax.plot(deltas, '^-', label='TD Residual ($\\lambda=0$)', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Time step $t$')
ax.set_ylabel('Advantage')
ax.set_title('Advantage Estimates: Bias-Variance Tradeoff')
ax.legend()

plt.tight_layout()
plt.show()

**Verification**: The assertions above confirm two critical GAE identities:

1. **GAE($\lambda{=}1$) = Monte Carlo advantage** $\hat{A}_t = \sum_{l=0}^{T-t-1} \gamma^l r_{t+l} + \gamma^{T-t} V(s_T) - V(s_t)$. This is the zero-bias, high-variance extreme.
2. **GAE($\lambda{=}0$) = one-step TD error** $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$. This is the high-bias, low-variance extreme.

The $\lambda$ parameter interpolates between these extremes. Schulman et al. (2016) show that the optimal $\lambda$ depends on the value function approximation quality: better critics allow lower $\lambda$ (less variance) without introducing bias.

In [ ]:
# ── GAE Computation Assertions ────────────────────────────────────────

# Returns must be finite
returns_arr = np.array(buf.returns)
assert np.all(np.isfinite(returns_arr)), "GAE returns contain NaN/Inf"

# Advantages must be finite
adv_arr = np.array(buf.advantages)
assert np.all(np.isfinite(adv_arr)), "GAE advantages contain NaN/Inf"

# Normalized advantages should have near-zero mean and unit std
if len(adv_arr) > 1:
    adv_norm = (adv_arr - adv_arr.mean()) / (adv_arr.std() + 1e-8)
    assert abs(adv_norm.mean()) < 1e-6, \
        f"Normalized advantages should have zero mean, got {adv_norm.mean():.2e}"
    assert abs(adv_norm.std() - 1.0) < 1e-6, \
        f"Normalized advantages should have unit std, got {adv_norm.std():.6f}"

# GAE manual vs buffer match (from cell above)
assert max_diff < 1e-10, \
    f"Manual and buffer GAE computation disagree by {max_diff:.2e}"

# GAE(λ=1) must equal MC advantage
mc_diff = np.max(np.abs(gae_lam1 - mc_advantage))
assert mc_diff < 1e-10, f"GAE(λ=1) ≠ MC advantage, max diff = {mc_diff:.2e}"

# GAE(λ=0) must equal one-step TD residual
td_diff = np.max(np.abs(gae_lam0 - deltas))
assert td_diff < 1e-10, f"GAE(λ=0) ≠ TD residual, max diff = {td_diff:.2e}"

# MC returns should be finite
assert np.all(np.isfinite(mc_returns)), "MC returns contain NaN/Inf"

print("✓ All GAE assertions passed")

## 4. PPO — Proximal Policy Optimization

PPO (Schulman et al., 2017) is the workhorse of modern policy-gradient RL.
It stabilizes training by preventing large policy updates.

### From Policy Gradients to Trust Regions

Recall from Section 2 that the **policy gradient theorem** gives us:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\!\left[\nabla_\theta \log \pi_\theta(a|s) \cdot \hat{A}(s, a)\right]$$

Vanilla policy gradient (REINFORCE) takes steps proportional to this gradient,
but choosing the step size is treacherous: too large a step can catastrophically
collapse performance, and the "right" step size varies across training. This
motivates the **trust region** formulation.

**TRPO** (Schulman et al., 2015) recasts the update as a constrained optimization:

$$\max_\theta \;\; \mathbb{E}_{s, a \sim \pi_{\theta_\text{old}}}\!\left[\frac{\pi_\theta(a|s)}{\pi_{\theta_\text{old}}(a|s)} \hat{A}(s, a)\right] \quad \text{s.t.} \quad D_\text{KL}\!\left(\pi_{\theta_\text{old}} \,\|\, \pi_\theta\right) \le \delta$$

The KL constraint ensures the new policy stays "close" to the old one in
distribution space, not parameter space. TRPO solves this via a second-order
conjugate gradient method — effective but expensive ($O(n^2)$ in parameters)
and difficult to implement with shared actor-critic architectures.

**PPO's key insight**: replace the hard KL constraint with a simple clipping
mechanism that achieves a similar trust-region effect using only first-order
optimization. The resulting algorithm is nearly as stable as TRPO but as
simple to implement as vanilla policy gradient.

### Conservative Policy Iteration (CPI) surrogate

Define the probability ratio:

$$r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{\text{old}}}(a_t | s_t)}$$

The CPI objective is:

$$L^{\text{CPI}}(\theta) = \hat{\mathbb{E}}_t\left[r_t(\theta) \, \hat{A}_t\right]$$

This is exact when $\theta = \theta_{\text{old}}$ (where $r_t = 1$) and its
gradient equals the policy gradient at that point. However, naively maximizing
$L^{\text{CPI}}$ can lead to destructively large updates.

### Clipped surrogate objective

PPO clips the ratio to stay within $[1 - \epsilon, 1 + \epsilon]$:

$$\boxed{L^{\text{CLIP}}(\theta) = \hat{\mathbb{E}}_t\left[\min\!\left(
r_t(\theta) \hat{A}_t, \;
\text{clip}\left(r_t(\theta),\, 1-\epsilon,\, 1+\epsilon\right) \hat{A}_t
\right)\right]}$$

### Why clipping works

Consider two cases:

1. **$\hat{A}_t > 0$** (good action): The objective wants to *increase* $r_t$,
   but clipping caps the benefit at $r_t = 1 + \epsilon$.
   $$\min\!\left(r_t \hat{A}_t,\; (1+\epsilon) \hat{A}_t\right)$$

2. **$\hat{A}_t < 0$** (bad action): The objective wants to *decrease* $r_t$,
   but clipping caps the change at $r_t = 1 - \epsilon$.
   $$\min\!\left(r_t \hat{A}_t,\; (1-\epsilon) \hat{A}_t\right)$$

In both cases the `min` removes any incentive to move $r_t$ beyond the
clipping boundary, creating a **trust region** in probability-ratio space.

### Connection to KL divergence (why this prevents catastrophic updates)

TRPO (Schulman et al., 2015) directly constrains
$D_{\text{KL}}(\pi_{\theta_{\text{old}}} \| \pi_\theta) \le \delta$, which
requires a costly conjugate-gradient solve. PPO replaces this with the
clipping heuristic, which achieves a similar effect:

When $r_t \in [1-\epsilon, 1+\epsilon]$ for all $(s_t, a_t)$, the KL divergence
is bounded. For a single state-action pair with binary action space:

$$D_{\text{KL}} = p_{\text{old}} \log\frac{p_{\text{old}}}{p_{\text{new}}}
+ (1-p_{\text{old}}) \log\frac{1-p_{\text{old}}}{1-p_{\text{new}}}$$

If $p_{\text{new}} / p_{\text{old}} \in [1-\epsilon, 1+\epsilon]$ then
$D_{\text{KL}} = O(\epsilon^2)$ by Taylor expansion of $\log(1+x) \approx x - x^2/2$.
So **clipping at $\epsilon = 0.2$ implicitly bounds KL at $\approx 0.02$**, which
matches the TRPO default. The gradient is zero outside the clip, so the optimiser
naturally stays within this implicit trust region — no second-order solve needed.

### Full PPO loss

$$L(\theta) = L^{\text{CLIP}} - c_1 L^{\text{VF}} + c_2 H[\pi_\theta]$$

where $L^{\text{VF}} = \frac{1}{2}\|V_\theta(s) - R_t\|^2$ is the value loss
and $H$ is the entropy bonus encouraging exploration.

In [ ]:
# ── Visualize the PPO clipping function ────────────────────────────────

eps = 0.2
r = np.linspace(0.0, 2.5, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, A_sign, title in zip(axes, [1.0, -1.0],
                              ['Positive Advantage ($\hat{A} > 0$)',
                               'Negative Advantage ($\hat{A} < 0$)']):
    A = A_sign
    L_cpi   = r * A
    r_clip  = np.clip(r, 1 - eps, 1 + eps)
    L_clip  = np.minimum(r * A, r_clip * A)

    ax.plot(r, L_cpi, '--', label='$L^{CPI} = r \\hat{A}$', alpha=0.6)
    ax.plot(r, r_clip * A, ':', label='Clipped term', alpha=0.6)
    ax.plot(r, L_clip, linewidth=2.5, label='$L^{CLIP}$')

    ax.axvline(1 - eps, color='gray', linestyle=':', alpha=0.4)
    ax.axvline(1 + eps, color='gray', linestyle=':', alpha=0.4)
    ax.axvline(1.0, color='black', linestyle='-', alpha=0.3, linewidth=0.8)

    ax.fill_betweenx([-2, 3], 1 - eps, 1 + eps, alpha=0.07, color='green',
                     label=f'Trust region $[{1-eps}, {1+eps}]$')

    ax.set_xlabel('Probability Ratio $r_t(\\theta)$')
    ax.set_ylabel('Objective $L$')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 2.5)
    ax.set_ylim(-1.8, 1.8)

plt.suptitle('PPO Clipped Surrogate Objective', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Train PPO on DroneHoverEnv ─────────────────────────────────────────

env = DroneHoverEnv(target_pos=np.array([0.0, 0.0, 1.0]), max_steps=200, dt=0.02, seed=42)
ppo = PPO(obs_dim=12, act_dim=4, lr_policy=5e-4, lr_value=1e-3,
          gamma=0.99, lam=0.95, clip_eps=0.2, entropy_coef=0.01,
          n_epochs=4, batch_size=64, seed=42)

n_episodes = 60
rollout_steps = 200

ppo_rewards = []
ppo_pos_errors = []
ppo_policy_losses = []
ppo_value_losses = []

t0 = time.time()
for ep in range(n_episodes):
    obs = env.reset()
    ep_reward = 0.0
    ep_pos_errors = []

    for step in range(rollout_steps):
        action, log_prob, value = ppo.select_action(obs)
        action_clipped = np.clip(action, -1, 1)
        next_obs, reward, done, info = env.step(action_clipped)

        ppo.store_transition(obs, action_clipped, reward, done, log_prob, value)
        ep_reward += reward
        ep_pos_errors.append(np.linalg.norm(obs[:3]))

        obs = next_obs
        if done:
            obs = env.reset()

    stats = ppo.update(obs)
    ppo_rewards.append(ep_reward)
    ppo_pos_errors.append(np.mean(ep_pos_errors))
    ppo_policy_losses.append(stats.get('policy_loss', 0))
    ppo_value_losses.append(stats.get('value_loss', 0))

    if (ep + 1) % 20 == 0:
        print(f"Episode {ep+1:3d} | Return: {ep_reward:8.2f} | "
              f"Pos Err: {ppo_pos_errors[-1]:.3f} | "
              f"P Loss: {stats.get('policy_loss',0):.4f} | "
              f"V Loss: {stats.get('value_loss',0):.4f}")

elapsed = time.time() - t0
print(f"\nTraining time: {elapsed:.1f}s")
print(f"Final avg return (last 10): {np.mean(ppo_rewards[-10:]):.2f}")
print(f"Final avg pos error (last 10): {np.mean(ppo_pos_errors[-10:]):.3f} m")

# NOTE: This pure-NumPy PPO uses finite-difference gradients, updating only
# ~10 randomly sampled parameters per weight matrix per batch.  With ~5000
# total parameters, this means <2% coverage per update — far too sparse
# for meaningful learning.  The learning curves below will be mostly flat.
#
# In production, PyTorch/JAX backpropagation computes exact gradients for
# ALL parameters simultaneously, enabling convergence in ~200-500 episodes
# for this hover task.  The purpose here is to demonstrate the PPO algorithm
# structure (rollout collection, advantage estimation, clipped objective)
# correctly, not to achieve convergence with numerical gradients.
print("\n⚠ Finite-difference gradients update <2% of parameters per step.")
print("  With backpropagation (PyTorch), this task converges in ~300 episodes.")

def running_avg(arr, window=10):
    out = np.empty(len(arr))
    for i in range(len(arr)):
        out[i] = np.mean(arr[max(0, i - window + 1):i + 1])
    return out

def running_std(arr, window=10):
    out = np.empty(len(arr))
    for i in range(len(arr)):
        segment = arr[max(0, i - window + 1):i + 1]
        out[i] = np.std(segment) if len(segment) > 1 else 0.0
    return out

eps = np.arange(1, n_episodes + 1)

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

# ── Top-left: Returns with confidence band + dual-axis policy loss ───
ax1 = fig.add_subplot(gs[0, 0])
r_mean = running_avg(ppo_rewards)
r_std = running_std(ppo_rewards)
ax1.fill_between(eps, r_mean - r_std, r_mean + r_std,
                 color='#3498db', alpha=0.15, label='±1 std (rolling)')
ax1.plot(eps, ppo_rewards, color='#bdc3c7', alpha=0.4, lw=0.8)
ax1.plot(eps, r_mean, color='#2c3e50', lw=2.2, label='Mean return')
if len(ppo_rewards) > 20:
    converge_val = np.mean(ppo_rewards[-10:])
    ax1.axhspan(converge_val - np.std(ppo_rewards[-10:]),
                converge_val + np.std(ppo_rewards[-10:]),
                alpha=0.08, color='green', label='Convergence region')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Episode Return', color='#2c3e50')
ax1.set_title('PPO: Returns & Policy Loss', fontweight='bold')

ax1r = ax1.twinx()
ax1r.plot(eps, running_avg(ppo_policy_losses), color='#e74c3c',
          lw=1.5, ls='--', alpha=0.8, label='Policy loss')
ax1r.set_ylabel('Policy Loss', color='#e74c3c')
ax1r.tick_params(axis='y', labelcolor='#e74c3c')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1r.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc='best')

# ── Top-right: Position error with confidence band ───────────────────
ax2 = fig.add_subplot(gs[0, 1])
pe_mean = running_avg(ppo_pos_errors)
pe_std = running_std(ppo_pos_errors)
ax2.fill_between(eps, np.maximum(pe_mean - pe_std, 0), pe_mean + pe_std,
                 color='#e67e22', alpha=0.15)
ax2.plot(eps, ppo_pos_errors, color='#bdc3c7', alpha=0.4, lw=0.8)
ax2.plot(eps, pe_mean, color='#d35400', lw=2.2, label='Mean pos error')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Position Error [m]')
ax2.set_title('PPO: Mean Position Error', fontweight='bold')
ax2.legend(fontsize=8)

# ── Bottom-left: Value loss ──────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(eps, ppo_value_losses, color='#bdc3c7', alpha=0.4, lw=0.8)
ax3.plot(eps, running_avg(ppo_value_losses), color='#8e44ad', lw=2.2,
         label='Value loss (smoothed)')
ax3.set_xlabel('Episode')
ax3.set_ylabel('Value Loss')
ax3.set_title('PPO: Critic Loss', fontweight='bold')
ax3.legend(fontsize=8)

# ── Bottom-right: Diagnostic — approximate clip fraction ─────────────
ax4 = fig.add_subplot(gs[1, 1])
norm_rewards = (np.array(ppo_rewards) - np.mean(ppo_rewards))
norm_rewards /= (np.std(ppo_rewards) + 1e-8)
policy_loss_arr = np.array(ppo_policy_losses)
ax4.scatter(norm_rewards, policy_loss_arr, c=eps, cmap='viridis',
            s=25, alpha=0.7, edgecolors='none')
cb = plt.colorbar(ax4.collections[0], ax=ax4, label='Episode')
ax4.set_xlabel('Normalized Return')
ax4.set_ylabel('Policy Loss')
ax4.set_title('PPO: Return vs Policy Loss Diagnostic', fontweight='bold')

fig.suptitle('PPO Training on DroneHoverEnv (finite-difference gradients)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Interpreting PPO results**: With finite-difference gradients (no autograd), PPO updates only ~2% of parameters per step — making convergence very slow. In production (PyTorch + GPU), PPO converges in minutes, not hours. The key insight is the **clipping mechanism**: by bounding the policy ratio $r(\theta) = \pi_\theta(a|s) / \pi_{\theta_\text{old}}(a|s)$ to $[1-\epsilon, 1+\epsilon]$, PPO creates an implicit **trust region** that prevents catastrophic policy updates — critical for stable quadrotor control where a single bad update can cause a crash.

> **SCR-PPO (2026)**: Recent work by Li et al. adds a *Lipschitz-constrained regularizer* to the PPO objective, penalizing policies whose action output changes too rapidly with state perturbations. This reduces high-frequency oscillations in rotor commands (typically 8–12 Hz) that cause real-world instability. The key addition to the loss: $\mathcal{L}_\text{SCR} = \lambda_L \max(0, \|\nabla_s \pi_\theta(s)\|_F - L_\max)$ where $L_\max$ is the Lipschitz bound.

### PPO Training Diagnostics: What to Monitor

When training PPO for quadrotor control (or any continuous-action task),
four diagnostic signals are critical for detecting and fixing training
pathologies:

#### 1. Episode Return (top-left plot)

The most important metric. A healthy training run shows:
- **Early phase**: returns fluctuate widely as the policy explores
- **Learning phase**: the rolling mean trends upward with decreasing variance
- **Convergence**: returns plateau within a "convergence band" (green shaded region)

**Red flags**:
- Returns *decrease* after initial improvement → learning rate too high or
  clipping threshold $\epsilon$ too large (policy updates are overshooting)
- Returns are flat from the start → reward scale too small, learning rate
  too low, or exploration is insufficient (try increasing entropy coefficient)
- Returns oscillate without trend → GAE $\lambda$ may be poorly tuned;
  try $\lambda = 0.95$ as a starting point

#### 2. Position Error (top-right)

For hover tasks, position error should decrease monotonically if the policy
is improving. A divergence between return improvement and position error
can indicate **reward hacking** — the policy optimises a reward component
(e.g., minimising angular velocity) at the expense of the primary objective
(position tracking).

#### 3. Policy Loss (dual-axis, top-left)

The PPO clipped surrogate loss has a distinctive signature:
- **Magnitude**: typically $|L^{\text{CLIP}}| \in [10^{-3}, 10^{-1}]$. Values
  near zero mean the advantage estimates are small (good policy, small updates).
  Very large values suggest the advantages are poorly normalised.
- **Sign**: the loss should be *negative* (we maximise the objective, which is
  equivalent to minimising the negative). Persistent positive loss indicates
  the policy is worsening.
- **Correlation with returns**: loss and returns should be loosely anti-correlated.
  The bottom-right diagnostic scatter plot visualises this relationship — a
  clear negative slope indicates healthy learning.

#### 4. Value Loss (bottom-left)

The critic's MSE loss reflects how well the value function approximates
$V^\pi(s)$. Ideally:
- Value loss *decreases* over training as the critic becomes more accurate
- A sudden spike in value loss often precedes a policy collapse (the critic
  cannot predict returns for the new, different policy)
- Very low value loss + flat returns → the critic has converged but the
  policy is stuck; increase the policy learning rate or entropy bonus

#### Hyperparameter Sensitivity for Drone Hover

| Hyperparameter | Recommended | Effect of Increasing | Effect of Decreasing |
|:---------------|:------------|:---------------------|:---------------------|
| $\epsilon$ (clip) | 0.2 | Larger updates, less stable | Smaller updates, slower but safer |
| $\lambda$ (GAE) | 0.95 | Lower bias, higher variance | Higher bias, lower variance |
| $\gamma$ (discount) | 0.99 | Values long-term rewards, slower learning | Myopic, faster but may miss optimal hover |
| Entropy coeff. | 0.01 | More exploration, noisier actions | Less exploration, faster convergence if near optimal |
| Learning rate | $5 \times 10^{-4}$ | Faster but risk overshooting | Slower but more stable |
| Batch size | 2048+ | More stable gradients, slower wall-clock | Noisier gradients, faster per-episode |
| N epochs | 4–10 | Better data utilisation, risk overfitting to batch | Less overfitting, wasted data |

> **Quick-start recipe for quadrotor hover**: Start with $\epsilon = 0.2$,
> $\lambda = 0.95$, $\gamma = 0.99$, entropy $= 0.01$, batch $= 4096$,
> 10 PPO epochs, learning rate $3 \times 10^{-4}$ with linear decay.
> This matches the SimpleFlight (2025) configuration that achieves
> zero-shot sim-to-real transfer on Crazyflie.

In [ ]:
# ── PPO Training Assertions ───────────────────────────────────────────

# Policy losses must be finite
for i, pl in enumerate(ppo_policy_losses):
    assert np.isfinite(pl), f"PPO policy_loss at episode {i} is NaN/Inf: {pl}"

# Value losses must be finite and non-negative
for i, vl in enumerate(ppo_value_losses):
    assert np.isfinite(vl), f"PPO value_loss at episode {i} is NaN/Inf: {vl}"
    assert vl >= 0, f"PPO value_loss at episode {i} is negative: {vl}"

# Episode rewards must be finite
for i, r in enumerate(ppo_rewards):
    assert np.isfinite(r), f"PPO return at episode {i} is NaN/Inf: {r}"

# Position errors must be non-negative and finite
for i, pe in enumerate(ppo_pos_errors):
    assert np.isfinite(pe), f"PPO pos_error at episode {i} is NaN/Inf"
    assert pe >= 0, f"PPO pos_error at episode {i} is negative: {pe}"

# Clipping fraction sanity: policy losses should not explode
assert all(abs(pl) < 1e6 for pl in ppo_policy_losses), \
    "PPO policy loss magnitude exceeded 1e6 — possible numerical instability"

print(f"✓ All PPO assertions passed ({len(ppo_rewards)} episodes verified)")

## 5. SAC — Soft Actor-Critic

SAC (Haarnoja et al., 2018) augments the standard RL objective with an
**entropy bonus**, yielding policies that are both performant and exploratory.

### Maximum entropy objective

$$J(\pi) = \sum_{t=0}^T \mathbb{E}_{(s_t, a_t) \sim \rho_\pi}\left[
r(s_t, a_t) + \alpha \, \mathcal{H}[\pi(\cdot|s_t)]
\right]$$

where the entropy term $\mathcal{H}[\pi(\cdot|s)] = -\mathbb{E}_{a \sim \pi}[\log \pi(a|s)]$
and $\alpha > 0$ is the *temperature* controlling exploration-exploitation trade-off.

### Soft Bellman equation

The soft state-value function satisfies:

$$V(s) = \mathbb{E}_{a \sim \pi}\left[Q(s, a) - \alpha \log \pi(a|s)\right]$$

and the soft Q-function obeys:

$$Q(s, a) = r(s, a) + \gamma \, \mathbb{E}_{s' \sim P}\left[V(s')\right]
= r + \gamma \, \mathbb{E}_{s'}\left[
\mathbb{E}_{a' \sim \pi}\left[Q(s', a') - \alpha \log \pi(a'|s')\right]
\right]$$

### Reparameterization trick

To differentiate through the stochastic policy, we reparameterize:

$$a = f_\theta(\epsilon; s) = \tanh\!\left(\mu_\theta(s) + \sigma_\theta(s) \cdot \epsilon\right),
\quad \epsilon \sim \mathcal{N}(0, I)$$

The $\tanh$ squashing ensures bounded actions $a \in (-1, 1)^4$.

### Squashed Gaussian log-probability

When applying $\tanh$, the change-of-variables formula adjusts the
density. Let $u = \mu + \sigma \epsilon$ (pre-squash) and $a = \tanh(u)$:

$$\log \pi(a|s) = \log \mathcal{N}(u; \mu_\theta, \sigma_\theta)
- \sum_{i=1}^{\dim(a)} \log\!\left(1 - \tanh^2(u_i)\right)$$

The correction term $-\sum \log(1 - \tanh^2(u_i))$ accounts for the
Jacobian of the $\tanh$ transform.

### Clipped double-Q trick

To combat Q-function overestimation, SAC maintains two independent
Q-networks and uses their minimum as the target:

$$y = r + \gamma \left(\min\!\left(Q_{\phi_1}'(s', a'),\, Q_{\phi_2}'(s', a')\right)
- \alpha \log \pi_\theta(a'|s')\right)$$

where $Q'$ are *target networks* updated via Polyak averaging:
$\phi' \leftarrow \tau \phi + (1 - \tau) \phi'$.

### Automatic temperature tuning

The temperature $\alpha$ is critical: too high → random policy (pure
exploration), too low → deterministic policy (no exploration). SAC v2
learns $\alpha$ automatically by solving a constrained optimization:

$$\min_\alpha \; \mathbb{E}_{a \sim \pi_\theta}\left[-\alpha \log \pi_\theta(a|s) - \alpha \bar{\mathcal{H}}\right]$$

where $\bar{\mathcal{H}} = -\dim(\mathcal{A})$ is the target entropy
(one nat per action dimension). The gradient is:

$$\nabla_\alpha J(\alpha) = -\mathbb{E}\left[\log \pi_\theta(a|s) + \bar{\mathcal{H}}\right]$$

**Intuition**: if the policy is more deterministic than the target
($\mathcal{H}[\pi] < \bar{\mathcal{H}}$), $\alpha$ increases to encourage
exploration. If the policy is too random, $\alpha$ decreases.

### Why entropy matters for drone control

1. **Multi-modal solutions**: Different obstacle-avoidance strategies
   (go left vs go right) are equally valid — entropy prevents premature
   commitment to one mode.
2. **Robustness**: High-entropy policies explore more of the state space
   during training, discovering failure modes early.
3. **Graceful degradation**: Under domain shift (sim → real), an entropy-
   regularized policy defaults to cautious, diverse behavior rather than
   confidently executing a wrong plan.

### When SAC Outperforms PPO for Drone Control

The choice between SAC and PPO is not universal — it depends on the deployment context:

| Scenario | Preferred | Rationale |
|:---------|:----------|:----------|
| Continuous fine motor control | **SAC** | Entropy bonus explores the full continuous action manifold; PPO's Gaussian can collapse prematurely |
| Sample-limited sim budget | **SAC** | Off-policy replay buffer reuses every transition; PPO discards data after each update |
| High-dimensional actions ($\dim(\mathcal{A}) > 6$) | **SAC** | Reparameterization trick scales gracefully; PPO's log-prob gradient variance grows with $\dim(\mathcal{A})$ |
| Safety-critical deployment | **PPO** | Clipping provides tighter implicit trust region; lower steady-state variance |
| Sim-to-real transfer | **PPO** | Deterministic eval policy is more robust to distribution shift (SimpleFlight, 2025) |
| Multi-objective reward | **SAC** | Entropy regularization prevents mode collapse when the reward landscape is multi-modal |

### Deriving the Soft Policy Improvement Guarantee

The maximum entropy framework provides a formal guarantee that each SAC
policy update improves performance. Define the soft state-value under a
new policy $\pi'$:

$$V^{\pi'}(s) = \mathbb{E}_{a \sim \pi'}\!\left[Q^{\pi}(s, a) - \alpha \log \pi'(a|s)\right]$$

If we choose $\pi'$ to maximise the RHS (a soft policy improvement step):

$$\pi_{\text{new}} = \arg\min_{\pi'} D_\text{KL}\!\left(\pi'(\cdot|s) \;\Big\|\; \frac{\exp\!\left(\frac{1}{\alpha}Q^{\pi_\text{old}}(s, \cdot)\right)}{Z(s)}\right)$$

then $Q^{\pi_\text{new}}(s, a) \ge Q^{\pi_\text{old}}(s, a)$ for all $(s, a)$
(Haarnoja et al., 2018, Lemma 1). Repeated application converges to the
optimal soft policy $\pi^*$ — the unique fixed point of the soft Bellman
operator.

In [ ]:
# ── Train SAC on DroneHoverEnv ─────────────────────────────────────────

env_sac = DroneHoverEnv(target_pos=np.array([0.0, 0.0, 1.0]), max_steps=200, dt=0.02, seed=42)
sac = SAC(obs_dim=12, act_dim=4, gamma=0.99, tau=0.005, alpha=0.2,
          lr=3e-4, buffer_size=50000, batch_size=128, seed=42)

n_episodes_sac = 60
sac_rewards = []
sac_pos_errors = []
sac_q_losses = []

t0 = time.time()
for ep in range(n_episodes_sac):
    obs = env_sac.reset()
    ep_reward = 0.0
    ep_pos_errors = []
    done = False

    while not done:
        action = sac.select_action(obs)
        next_obs, reward, done, info = env_sac.step(action)
        sac.store_transition(obs, action, reward, next_obs, done)

        stats = sac.update()
        ep_reward += reward
        ep_pos_errors.append(np.linalg.norm(obs[:3]))
        obs = next_obs

    sac_rewards.append(ep_reward)
    sac_pos_errors.append(np.mean(ep_pos_errors) if ep_pos_errors else 0)
    sac_q_losses.append(stats.get('q_loss', 0) if stats else 0)

    if (ep + 1) % 20 == 0:
        print(f"Episode {ep+1:3d} | Return: {ep_reward:8.2f} | "
              f"Pos Err: {sac_pos_errors[-1]:.3f} | "
              f"Q Loss: {sac_q_losses[-1]:.4f}")

elapsed = time.time() - t0
print(f"\nTraining time: {elapsed:.1f}s")
print(f"Final avg return (last 10): {np.mean(sac_rewards[-10:]):.2f}")
print(f"Final avg pos error (last 10): {np.mean(sac_pos_errors[-10:]):.3f} m")

# Compare PPO vs SAC learning curves
eps = np.arange(1, n_episodes + 1)
c_ppo, c_sac = '#2980b9', '#e74c3c'

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.35)

# ── (1) Episode returns with confidence bands ────────────────────────
ax = fig.add_subplot(gs[0, 0])
for data, color, label in [(ppo_rewards, c_ppo, 'PPO'), (sac_rewards, c_sac, 'SAC')]:
    mean = running_avg(data)
    std = running_std(data)
    ax.fill_between(eps, mean - std, mean + std, color=color, alpha=0.12)
    ax.plot(eps, data, color=color, alpha=0.25, lw=0.8)
    ax.plot(eps, mean, color=color, lw=2.2, label=label)
ax.set_xlabel('Episode'); ax.set_ylabel('Return')
ax.set_title('Episode Returns', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# ── (2) Tracking quality ─────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
for data, color, label in [(ppo_pos_errors, c_ppo, 'PPO'), (sac_pos_errors, c_sac, 'SAC')]:
    mean = running_avg(data)
    std = running_std(data)
    ax.fill_between(eps, np.maximum(mean - std, 0), mean + std, color=color, alpha=0.12)
    ax.plot(eps, data, color=color, alpha=0.25, lw=0.8)
    ax.plot(eps, mean, color=color, lw=2.2, label=label)
ax.set_xlabel('Episode'); ax.set_ylabel('Position Error [m]')
ax.set_title('Tracking Quality', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# ── (3) Sample efficiency — cumulative return vs. env steps ──────────
ax = fig.add_subplot(gs[0, 2])
ppo_steps = eps * 200
sac_steps = eps * 200
ppo_cumret = np.cumsum(ppo_rewards)
sac_cumret = np.cumsum(sac_rewards)
ax.plot(ppo_steps, ppo_cumret, color=c_ppo, lw=2.2, label='PPO')
ax.plot(sac_steps, sac_cumret, color=c_sac, lw=2.2, label='SAC')
ax.fill_between(ppo_steps, ppo_cumret, sac_cumret, where=sac_cumret > ppo_cumret,
                color=c_sac, alpha=0.08, label='SAC advantage')
ax.fill_between(ppo_steps, ppo_cumret, sac_cumret, where=ppo_cumret > sac_cumret,
                color=c_ppo, alpha=0.08, label='PPO advantage')
ax.set_xlabel('Total Environment Steps'); ax.set_ylabel('Cumulative Return')
ax.set_title('Sample Efficiency', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── (4) 3D Trajectories overlay (last eval episode each) ────────────
ax3d = fig.add_subplot(gs[1, 0], projection='3d')
eval_env_cmp = DroneHoverEnv(target_pos=np.array([0., 0., 1.]),
                              max_steps=200, dt=0.02, seed=77)
for agent_fn, color, label in [
    (lambda o: np.clip(ppo.select_action(o)[0], -1, 1), c_ppo, 'PPO'),
    (lambda o: sac.select_action(o, deterministic=True), c_sac, 'SAC'),
]:
    obs_cmp = eval_env_cmp.reset()
    traj_cmp = [eval_env_cmp.state.position.copy()]
    for _ in range(200):
        obs_cmp, _, done_cmp, _ = eval_env_cmp.step(agent_fn(obs_cmp))
        traj_cmp.append(eval_env_cmp.state.position.copy())
        if done_cmp:
            break
    traj_cmp = np.array(traj_cmp)
    ax3d.plot(traj_cmp[:, 0], traj_cmp[:, 1], traj_cmp[:, 2],
              color=color, lw=1.8, alpha=0.85, label=label)
    ax3d.scatter(*traj_cmp[0], color=color, s=40, marker='o', zorder=5)
    ax3d.scatter(*traj_cmp[-1], color=color, s=40, marker='x', zorder=5)
ax3d.scatter(*eval_env_cmp.target_pos, color='gold', s=200, marker='*',
             edgecolors='k', linewidths=1, zorder=10, label='Target')
ax3d.set_xlabel('X [m]'); ax3d.set_ylabel('Y [m]'); ax3d.set_zlabel('Z [m]')
ax3d.set_title('3D Trajectory Overlay', fontweight='bold')
ax3d.legend(fontsize=7, loc='upper right')

# ── (5) Action distribution histograms (converged policies) ──────────
ax_h1 = fig.add_subplot(gs[1, 1])
ppo_actions_hist = []
sac_actions_hist = []
eval_env_h = DroneHoverEnv(target_pos=np.array([0., 0., 1.]),
                            max_steps=200, dt=0.02, seed=88)
obs_h = eval_env_h.reset()
for _ in range(200):
    a_ppo = np.clip(ppo.select_action(obs_h)[0], -1, 1)
    ppo_actions_hist.append(a_ppo.copy())
    obs_h, _, done_h, _ = eval_env_h.step(a_ppo)
    if done_h:
        break
obs_h = eval_env_h.reset()
for _ in range(200):
    a_sac = sac.select_action(obs_h, deterministic=False)
    sac_actions_hist.append(a_sac.copy())
    obs_h, _, done_h, _ = eval_env_h.step(a_sac)
    if done_h:
        break
ppo_actions_hist = np.array(ppo_actions_hist).flatten()
sac_actions_hist = np.array(sac_actions_hist).flatten()
bins = np.linspace(-1, 1, 40)
ax_h1.hist(ppo_actions_hist, bins=bins, alpha=0.5, color=c_ppo,
           label=f'PPO (σ={np.std(ppo_actions_hist):.2f})', density=True)
ax_h1.hist(sac_actions_hist, bins=bins, alpha=0.5, color=c_sac,
           label=f'SAC (σ={np.std(sac_actions_hist):.2f})', density=True)
ax_h1.set_xlabel('Action Value'); ax_h1.set_ylabel('Density')
ax_h1.set_title('Action Distribution (all motors)', fontweight='bold')
ax_h1.legend(fontsize=8)

# ── (6) Per-motor action comparison ──────────────────────────────────
ax_m = fig.add_subplot(gs[1, 2])
ppo_a2 = []
sac_a2 = []
eval_env_m = DroneHoverEnv(target_pos=np.array([0., 0., 1.]),
                            max_steps=200, dt=0.02, seed=88)
obs_m = eval_env_m.reset()
for _ in range(200):
    a_ = np.clip(ppo.select_action(obs_m)[0], -1, 1)
    ppo_a2.append(a_.copy())
    obs_m, _, d_, _ = eval_env_m.step(a_)
    if d_: break
obs_m = eval_env_m.reset()
for _ in range(200):
    a_ = sac.select_action(obs_m, deterministic=True)
    sac_a2.append(a_.copy())
    obs_m, _, d_, _ = eval_env_m.step(a_)
    if d_: break
ppo_a2 = np.array(ppo_a2)
sac_a2 = np.array(sac_a2)
steps_p = np.arange(len(ppo_a2))
steps_s = np.arange(len(sac_a2))
for i in range(min(4, ppo_a2.shape[1])):
    ax_m.plot(steps_p, ppo_a2[:, i], color=c_ppo, alpha=0.4, lw=0.7)
    if i < sac_a2.shape[1]:
        ax_m.plot(steps_s, sac_a2[:, i], color=c_sac, alpha=0.4, lw=0.7)
ax_m.plot([], [], color=c_ppo, lw=2, label='PPO motors')
ax_m.plot([], [], color=c_sac, lw=2, label='SAC motors')
ax_m.set_xlabel('Step'); ax_m.set_ylabel('Action')
ax_m.set_title('Per-Motor Commands Over Time', fontweight='bold')
ax_m.legend(fontsize=8)

fig.suptitle('PPO vs SAC Comparison on DroneHoverEnv',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**SAC vs PPO trade-off** (Kim et al., IEEE IISEC 2026): A systematic comparison across hover, waypoint tracking, and aggressive maneuvers finds:

| Criterion | PPO | SAC |
|-----------|-----|-----|
| Convergence speed | Slower (on-policy) | Faster (replay buffer) |
| Steady-state variance | **Lower** (clipping stabilizes) | Higher (entropy bonus explores) |
| Robustness to sensor noise | **Better** | Sensitive to $\alpha$ tuning |
| Sample efficiency | Worse (no replay) | **Better** (off-policy) |

**Practical rule**: Use **SAC for rapid prototyping** in simulation (faster convergence), but switch to **PPO for deployment** (lower variance, more stable sim-to-real transfer). This matches the SimpleFlight finding that PPO-based policies transfer better to real Crazyflie hardware.

In [ ]:
# ── SAC Training Assertions ───────────────────────────────────────────

# Q-losses must be finite
for i, ql in enumerate(sac_q_losses):
    assert np.isfinite(ql), f"SAC Q-loss at episode {i} is NaN/Inf: {ql}"
    assert ql >= 0, f"SAC Q-loss at episode {i} is negative: {ql}"

# Episode rewards must be finite
for i, r in enumerate(sac_rewards):
    assert np.isfinite(r), f"SAC return at episode {i} is NaN/Inf: {r}"

# Position errors must be non-negative and finite
for i, pe in enumerate(sac_pos_errors):
    assert np.isfinite(pe), f"SAC pos_error at episode {i} is NaN/Inf"
    assert pe >= 0, f"SAC pos_error at episode {i} is negative: {pe}"

# Entropy temperature alpha must be positive
assert sac.alpha > 0, f"SAC alpha must be positive, got {sac.alpha}"

# Q-values at a sample point should be finite
test_obs = env_sac.reset()
test_action = sac.select_action(test_obs, deterministic=True)
sa = np.concatenate([test_obs, test_action])
q1_test = sac.q1.forward(sa.reshape(1, -1))
q2_test = sac.q2.forward(sa.reshape(1, -1))
assert np.all(np.isfinite(q1_test)), "SAC Q1 value is NaN/Inf"
assert np.all(np.isfinite(q2_test)), "SAC Q2 value is NaN/Inf"

print(f"✓ All SAC assertions passed ({len(sac_rewards)} episodes, α={sac.alpha:.4f})")

## 6. Reward Engineering

Reward design is arguably the most critical component of applied drone RL.
A poorly designed reward can lead to reward hacking, instability, or
extremely slow learning.

### Quadratic tracking cost

The standard dense reward for hover stabilization is a weighted negative
quadratic:

$$r = -w_p \|p - p_d\|^2 - w_v \|v\|^2 - w_\omega \|\omega\|^2 - w_a \|a\|^2 + r_{\text{alive}}$$

Each term serves a purpose:
- $\|p - p_d\|^2$: position tracking (primary objective)
- $\|v\|^2$: penalizes high velocities (smooth behavior)
- $\|\omega\|^2$: penalizes aggressive rotation (smooth attitude)
- $\|a\|^2$: actuator regularization (energy efficiency)
- $r_{\text{alive}}$: survival bonus (discourages early termination)

### Sparse rewards

$$r_{\text{sparse}} = \begin{cases} +1 & \text{if } \|p - p_d\| < \epsilon \\ 0 & \text{otherwise} \end{cases}$$

Hard to learn but can yield sharper final behavior.

### Potential-based reward shaping

Ng et al. (1999) showed that adding a *potential-based* shaping term
preserves the optimal policy:

$$r'(s, a, s') = r(s, a) + \gamma \Phi(s') - \Phi(s)$$

A natural potential for the hover task is $\Phi(s) = -c \|p - p_d\|$,
which creates a shaped reward that guides the agent toward the goal without
changing the optimal solution.

In [ ]:
# ── Compare different reward designs ──────────────────────────────────

def run_ppo_with_reward(reward_fn, label, n_episodes=40, seed=42):
    """Train PPO with a custom reward function."""
    env = DroneHoverEnv(target_pos=np.array([0., 0., 1.]), max_steps=200, dt=0.02, seed=seed)
    agent = PPO(obs_dim=12, act_dim=4, lr_policy=5e-4, lr_value=1e-3,
                gamma=0.99, lam=0.95, clip_eps=0.2, n_epochs=3,
                batch_size=64, seed=seed)

    returns_hist = []
    for ep in range(n_episodes):
        obs = env.reset()
        ep_ret = 0.0
        for step in range(200):
            action, log_prob, value = agent.select_action(obs)
            action_clipped = np.clip(action, -1, 1)
            next_obs, _, done, _ = env.step(action_clipped)
            reward = reward_fn(obs, action_clipped, next_obs, env)
            agent.store_transition(obs, action_clipped, reward, done, log_prob, value)
            ep_ret += reward
            obs = next_obs
            if done:
                obs = env.reset()
        agent.update(obs)
        returns_hist.append(ep_ret)
    return returns_hist


# Reward 1: Dense quadratic
def reward_dense(obs, action, next_obs, env):
    pos_err = np.linalg.norm(obs[:3])
    vel_err = np.linalg.norm(obs[3:6])
    omega_err = np.linalg.norm(obs[9:12])
    return -1.0 * pos_err**2 - 0.1 * vel_err**2 - 0.01 * omega_err**2 - 0.001 * np.sum(action**2) + 0.1


# Reward 2: Sparse (only reward near target)
def reward_sparse(obs, action, next_obs, env):
    pos_err = np.linalg.norm(obs[:3])
    if pos_err < 0.2:
        return 1.0
    return -0.01


# Reward 3: Shaped (potential-based + dense)
def reward_shaped(obs, action, next_obs, env):
    pos_old = np.linalg.norm(obs[:3])
    pos_new = np.linalg.norm(next_obs[:3])
    shaping = 0.99 * (-pos_new) - (-pos_old)
    base = -0.5 * pos_old**2 - 0.05 * np.linalg.norm(obs[3:6])**2 + 0.05
    return base + 2.0 * shaping


print("Training with dense reward..."); t0 = time.time()
ret_dense  = run_ppo_with_reward(reward_dense,  'Dense')
print(f"  Done in {time.time()-t0:.1f}s")

print("Training with sparse reward...")
t0 = time.time()
ret_sparse = run_ppo_with_reward(reward_sparse, 'Sparse')
print(f"  Done in {time.time()-t0:.1f}s")

print("Training with shaped reward...")
t0 = time.time()
ret_shaped = run_ppo_with_reward(reward_shaped, 'Shaped')
print(f"  Done in {time.time()-t0:.1f}s")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(ret_dense,  label='Dense (quadratic)', alpha=0.7)
ax.plot(ret_sparse, label='Sparse (threshold)', alpha=0.7)
ax.plot(ret_shaped, label='Shaped (potential-based)', alpha=0.7)
ax.set_xlabel('Episode'); ax.set_ylabel('Return')
ax.set_title('Learning Curves: Different Reward Designs')
ax.legend()

ax = axes[1]
window = 5
for data, label in [(ret_dense, 'Dense'), (ret_sparse, 'Sparse'), (ret_shaped, 'Shaped')]:
    smoothed = np.convolve(data, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=label, alpha=0.8)
ax.set_xlabel('Episode'); ax.set_ylabel('Smoothed Return')
ax.set_title('Smoothed Learning Curves')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nFinal average return (last 10 episodes):")
print(f"  Dense:  {np.mean(ret_dense[-10:]):8.2f}")
print(f"  Sparse: {np.mean(ret_sparse[-10:]):8.2f}")
print(f"  Shaped: {np.mean(ret_shaped[-10:]):8.2f}")

### Understanding the Reward Landscape

The plots above visualise two critical aspects of reward design that directly
impact learning dynamics:

**1D reward profile (left)**: The quadratic dense reward creates a smooth
gradient everywhere — the agent always knows which direction reduces cost.
In contrast, the sparse reward provides *zero gradient* outside the
$\epsilon$-ball, meaning the agent must stumble into the goal region by
random exploration before receiving any learning signal. For quadrotor
hover, where the initial state can be meters from the target, sparse
rewards can require orders of magnitude more samples.

**2D reward heatmap (right)**: The concentric contours show that the
quadratic reward treats all directions equally — the agent is equally
penalised for being 1m to the left as 1m above the target. This
*isotropic* property is generally desirable for hover, but may not be
optimal for navigation tasks where horizontal deviations are safer than
vertical ones (ground collision risk).

#### Common Reward Design Pitfalls

| Pitfall | Symptom | Fix |
|:--------|:--------|:----|
| **Reward hacking** | Agent finds a high-reward state that isn't the intended goal | Add constraints or use potential-based shaping |
| **Reward scale mismatch** | One reward term dominates; others are ignored | Normalise each term to similar magnitude |
| **Dense reward too smooth** | Agent converges to "close enough" but not precise | Add a bonus for reaching the $\epsilon$-ball |
| **Sparse reward too hard** | Agent never finds the goal during exploration | Use curriculum learning (Section 10) or HER |
| **Action penalty too strong** | Agent learns to output near-zero actions (crashes) | Reduce $w_a$ or add a minimum-thrust bonus |

#### Potential-Based Shaping: Formal Guarantees

The key result of Ng et al. (1999) is that potential-based shaping
**preserves the set of optimal policies**. Formally, if we define:

$$r'(s, a, s') = r(s, a) + \gamma \Phi(s') - \Phi(s)$$

for any bounded function $\Phi: \mathcal{S} \to \mathbb{R}$, then every
optimal policy under $r'$ is also optimal under $r$ (and vice versa).
The intuition is that the shaping terms telescope over a trajectory:

$$\sum_{t=0}^{T} \left[\gamma \Phi(s_{t+1}) - \Phi(s_t)\right]
= \gamma^{T+1} \Phi(s_{T+1}) - \Phi(s_0)$$

This is a constant (independent of the policy), so it cannot change the
ranking of policies.

For drone hover, the natural potential $\Phi(s) = -c\|p - p_d\|$ provides
a "compass" that always points toward the goal, dramatically accelerating
early learning without distorting the optimal solution. The shaped reward
effectively gives the agent "credit" for moving closer to the target, even
if the per-step dense reward is still negative.

In [ ]:
# ── Reward Landscape Visualization ────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) Reward as a function of distance from goal (1D)
distances = np.linspace(0, 3.0, 200)
r_dense_curve = -1.0 * distances**2 + 0.1
r_sparse_curve = np.where(distances < 0.2, 1.0, -0.01)
r_shaped_curve = -0.5 * distances**2 + 2.0 * (-0.99 * np.gradient(distances) * 0) + 0.05
r_shaped_progress = -0.5 * distances**2 + 0.05

ax = axes[0]
ax.plot(distances, r_dense_curve, lw=2.2, color='#2980b9', label='Dense (quadratic)')
ax.plot(distances, r_sparse_curve, lw=2.2, color='#e74c3c', label='Sparse (threshold)')
ax.plot(distances, r_shaped_progress, lw=2.2, color='#27ae60', ls='--',
        label='Shaped (potential-based)')
ax.axvline(0.2, color='#e74c3c', alpha=0.3, ls=':', lw=1)
ax.annotate('Sparse threshold\n$\\|p-p_d\\| < \\epsilon$', xy=(0.2, 0.5),
            fontsize=8, ha='left', color='#e74c3c',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=0.8),
            xytext=(0.6, 0.7))
ax.set_xlabel('Distance from Goal $\\|p - p_d\\|$ [m]')
ax.set_ylabel('Reward $r(s, a)$')
ax.set_title('Reward vs. Distance (velocity=0, no action cost)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 3.0)

# (2) 2D heatmap of reward in the x-z plane (y=0, at hover)
xg = np.linspace(-2, 2, 100)
zg = np.linspace(-1, 3, 100)
X, Z = np.meshgrid(xg, zg)
target_xz = np.array([0.0, 1.0])
dist_xz = np.sqrt((X - target_xz[0])**2 + (Z - target_xz[1])**2)
reward_map = -1.0 * dist_xz**2 + 0.1

ax = axes[1]
im = ax.pcolormesh(X, Z, reward_map, cmap='RdYlGn', shading='auto')
ax.contour(X, Z, reward_map, levels=8, colors='k', linewidths=0.4, alpha=0.5)
ax.plot(0, 1, marker='*', color='gold', markersize=18,
        markeredgecolor='k', markeredgewidth=1.2, zorder=10, label='Target')
circle = plt.Circle((0, 1), 0.2, fill=False, color='white',
                     lw=1.5, ls='--', label='$\\epsilon$-ball')
ax.add_patch(circle)
plt.colorbar(im, ax=ax, label='Reward $r(s, a)$', shrink=0.85)
ax.set_xlabel('X [m]'); ax.set_ylabel('Z [m]')
ax.set_title('Reward Heatmap (X-Z Plane, y=0)', fontweight='bold')
ax.set_aspect('equal')
ax.legend(fontsize=8, loc='upper right')

plt.suptitle('Reward Landscape Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Navigation with Obstacles

Moving from hover stabilisation to goal-directed navigation introduces
a qualitatively different challenge: the drone must **perceive** its
surroundings and plan a collision-free path, all while maintaining stable
flight.

### Observation Space: Depth Rays as Perception

`DroneNavigationEnv` augments the base state with $K$ depth-ray readings:

$$s = [p - p_g, \; v, \; \phi, \theta, \psi, \; \omega, \; d_1, \ldots, d_K] \in \mathbb{R}^{12 + K}$$

where $d_k$ is the distance to the nearest obstacle along ray $k$, cast
from the drone's body frame at uniformly spaced azimuth angles.

**Why depth rays instead of raw images?** In this simplified environment,
depth rays serve as a proxy for the perception that a real system would
obtain from a depth camera or LiDAR. The key advantages:

- **Low-dimensional**: $K = 8$ rays vs. $640 \times 480 = 307{,}200$ depth
  pixels — orders of magnitude less data for the policy to process
- **Body-frame aligned**: the rays rotate with the drone, so the policy
  receives *egocentric* obstacle information that is invariant to global
  heading
- **Range-limited**: rays saturate at a maximum range $d_{\max}$,
  naturally encoding "no obstacle nearby" as a constant value

In a real deployment, one would replace depth rays with a learned encoder
that maps depth images to a compact representation (e.g., a variational
autoencoder or a pretrained ViT backbone), preserving the same information
in a form the RL policy can consume.

### Reward Design for Navigation

The navigation reward combines multiple terms, each addressing a different
aspect of the desired behavior:

| Term | Formula | Purpose |
|:-----|:--------|:--------|
| **Progress** | $r_{\text{prog}} = \|p_t - p_g\| - \|p_{t+1} - p_g\|$ | Reward for moving toward the goal |
| **Collision** | $-100$ on impact | Strong penalty to learn obstacle avoidance |
| **Goal** | $+100$ on arrival | Terminal reward for task completion |
| **Time** | $-0.01$ per step | Encourages efficiency (shorter paths) |
| **Action** | $-0.001 \|a\|^2$ | Smooth, energy-efficient commands |

The progress term is a **potential-based shaping** reward (Section 6) with
potential $\Phi(s) = -\|p - p_g\|$. It preserves the optimal policy while
providing dense signal that guides exploration toward the goal.

### The Exploration Problem in Navigation

Navigation exposes a fundamental tension in RL exploration:

- **Sparse goal reward**: the drone receives $+100$ only upon reaching the
  goal. With random exploration in 3D space, the probability of reaching a
  goal 5 meters away by chance is negligibly small.
- **Collision walls**: obstacles create "death zones" that terminate the
  episode with $-100$ penalty. Early in training, the agent collides
  frequently and never reaches the goal, receiving only negative feedback.
- **The progress term resolves this**: by rewarding any movement toward the
  goal, the agent receives useful gradient even when far away, while the
  depth rays provide the information needed to learn obstacle avoidance.

This illustrates a general principle: **dense rewards + rich observations
are prerequisites for RL to work in complex spatial tasks**. Without either,
the learning problem becomes prohibitively hard.

In [ ]:
# ── Train PPO on DroneNavigationEnv ───────────────────────────────────

n_depth_rays = 8
nav_env = DroneNavigationEnv(
    grid_shape=(30, 30, 15), resolution=0.3,
    n_obstacles=8, n_depth_rays=n_depth_rays,
    max_steps=300, dt=0.02, seed=42
)

obs_dim_nav = 12 + n_depth_rays
ppo_nav = PPO(obs_dim=obs_dim_nav, act_dim=4, lr_policy=5e-4, lr_value=1e-3,
              gamma=0.99, lam=0.95, clip_eps=0.2, n_epochs=3,
              batch_size=64, seed=42)

n_nav_episodes = 40
nav_rewards = []
nav_successes = []
nav_collisions = []
nav_lengths = []
nav_trajectories = []

t0 = time.time()
for ep in range(n_nav_episodes):
    obs = nav_env.reset()
    ep_reward = 0.0
    ep_len = 0
    done = False
    positions = [nav_env.state.position.copy()]
    goal_reached = False
    collided = False

    for step in range(300):
        action, log_prob, value = ppo_nav.select_action(obs)
        action_clipped = np.clip(action, -1, 1)
        next_obs, reward, done, info = nav_env.step(action_clipped)

        ppo_nav.store_transition(obs, action_clipped, reward, done, log_prob, value)
        ep_reward += reward
        ep_len += 1
        positions.append(nav_env.state.position.copy())

        if info.get('goal_reached', False):
            goal_reached = True
        if not nav_env.grid.is_free_world(nav_env.state.position):
            collided = True

        obs = next_obs
        if done:
            obs = nav_env.reset()
            break

    if not done:
        ppo_nav.update(obs)
    else:
        ppo_nav.update(next_obs)

    nav_rewards.append(ep_reward)
    nav_successes.append(float(goal_reached))
    nav_collisions.append(float(collided))
    nav_lengths.append(ep_len)
    if ep >= n_nav_episodes - 3:
        nav_trajectories.append(np.array(positions))

    if (ep + 1) % 10 == 0:
        recent_success = np.mean(nav_successes[max(0,ep-9):ep+1])
        recent_collision = np.mean(nav_collisions[max(0,ep-9):ep+1])
        print(f"Episode {ep+1:3d} | Return: {ep_reward:8.2f} | "
              f"Success: {recent_success:.0%} | "
              f"Collision: {recent_collision:.0%} | "
              f"Length: {ep_len}")

print(f"\nTraining time: {time.time()-t0:.1f}s")

# Plot navigation results
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

ax = axes[0, 0]
window = 5
success_smooth = np.convolve(nav_successes, np.ones(window)/window, mode='valid')
ax.plot(success_smooth)
ax.set_xlabel('Episode'); ax.set_ylabel('Success Rate')
ax.set_title('Navigation: Success Rate (smoothed)')
ax.set_ylim(-0.05, 1.05)

ax = axes[0, 1]
collision_smooth = np.convolve(nav_collisions, np.ones(window)/window, mode='valid')
ax.plot(collision_smooth, color='red')
ax.set_xlabel('Episode'); ax.set_ylabel('Collision Rate')
ax.set_title('Navigation: Collision Rate (smoothed)')
ax.set_ylim(-0.05, 1.05)

ax = axes[1, 0]
ax.plot(nav_lengths)
ax.set_xlabel('Episode'); ax.set_ylabel('Episode Length')
ax.set_title('Navigation: Episode Length')

ax = axes[1, 1]
ax.plot(nav_rewards)
ax.set_xlabel('Episode'); ax.set_ylabel('Return')
ax.set_title('Navigation: Episode Returns')

plt.tight_layout()
plt.show()

# 3D trajectory plot for last few episodes
if nav_trajectories:
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')
    for i, traj in enumerate(nav_trajectories):
        ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], '-', linewidth=1.5,
                alpha=0.8, label=f'Episode {n_nav_episodes - len(nav_trajectories) + i + 1}')
        ax.scatter(*traj[0], marker='o', s=50)
        ax.scatter(*traj[-1], marker='*', s=100)
    ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.set_zlabel('Z [m]')
    ax.set_title('Sample Navigation Trajectories')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 8. PPO vs SAC vs Classical Control

How do learned controllers compare with hand-tuned classical controllers?
This is not merely an academic question — the choice has direct implications
for safety certification, development cost, and deployment robustness.

### Systematic Comparison

| Property | PPO | SAC | PID / SE(3) |
|:---------|:----|:----|:------------|
| Learning | On-policy | Off-policy | No learning |
| Sample efficiency | Low (needs new data) | High (reuses data) | N/A |
| Stability | Stable (clipped updates) | Can be less stable | Guaranteed (Lyapunov) |
| Multimodal actions | No (Gaussian) | Yes (entropy + multimodal) | No |
| Model required | No | No | Yes (dynamics model) |
| Tuning | Hyperparameters | Hyperparameters + $\alpha$ | Controller gains |
| Robustness | Adapts if retrained | Adapts if retrained | Needs re-tuning |
| Sim-to-real | Domain randomization | Domain randomization | System identification |

### The Case for Classical Control

Despite the appeal of learned controllers, classical PID and SE(3) geometric
controllers remain dominant in production for good reason:

1. **Formal stability guarantees**: A well-tuned PID controller comes with
   Lyapunov-based stability proofs. For a hover task, we can *prove* that
   the drone will converge to the setpoint from any initial condition within
   a region of attraction. RL policies offer no such guarantees.

2. **Interpretability**: PID gains have physical meaning (proportional
   response, integral wind-up correction, derivative damping). When
   something goes wrong, engineers can diagnose and fix the issue. Neural
   network policies are opaque.

3. **Certification**: Aviation regulators (FAA, EASA) require that control
   software be certified to DO-178C standards. Currently, no neural-network
   controller has been certified for safety-critical flight control.

### The Case for RL

RL controllers excel in scenarios where classical approaches struggle:

1. **Unmodeled dynamics**: Wind gusts, payload changes, motor degradation —
   RL policies trained with domain randomization handle these naturally.

2. **Complex tasks**: Navigation through cluttered environments, aggressive
   maneuvers (racing, acrobatics), and multi-objective tasks are difficult
   to hand-code but natural for RL.

3. **Adaptive behavior**: RL policies can implicitly adapt to changing
   conditions within an episode (especially recurrent architectures like
   RAPTOR's adaptive policy).

### The Hybrid Consensus

The 2026 Atlantis Press review and the Zhang et al. (*IEEE TNNLS*, 2025)
survey both conclude that **hybrid architectures dominate real deployment**:

$$\underbrace{\text{RL policy}}_{\text{high-level}} \;\xrightarrow{\text{setpoints}}\; \underbrace{\text{classical controller}}_{\text{low-level}} \;\xrightarrow{\text{motor commands}}\; \text{actuators}$$

This architecture inherits the adaptability of RL at the planning level
and the stability guarantees of classical control at the actuation level.

### Robustness Under Perturbation

The evaluation below tests each controller under two conditions:
**nominal** (correct mass) and **perturbed** (mass increased by 20%).
This simulates a real-world scenario where the drone picks up a payload
or its mass changes due to battery consumption. The robustness test
reveals how gracefully each controller degrades under model mismatch.

In [ ]:
# ── Systematic comparison: PPO vs SAC vs PID on hover ────────────────

def evaluate_controller(controller_fn, env, n_eval_episodes=5, max_steps=200):
    """Evaluate a controller on the hover task and return metrics + trajectories."""
    all_returns = []
    all_pos_errors = []
    all_final_pos_errors = []
    all_trajectories = []

    for ep in range(n_eval_episodes):
        obs = env.reset()
        ep_reward = 0.0
        pos_errors = []
        trajectory = [env.state.position.copy() if hasattr(env, 'state') else obs[:3].copy()]
        done = False
        for step in range(max_steps):
            action = controller_fn(obs, env)
            obs, reward, done, _ = env.step(action)
            ep_reward += reward
            pos_errors.append(np.linalg.norm(obs[:3]))
            trajectory.append(env.state.position.copy() if hasattr(env, 'state') else obs[:3].copy())
            if done:
                break

        all_returns.append(ep_reward)
        all_pos_errors.append(np.mean(pos_errors))
        all_final_pos_errors.append(pos_errors[-1] if pos_errors else 0)
        all_trajectories.append(np.array(trajectory))

    return {
        'mean_return': np.mean(all_returns),
        'std_return': np.std(all_returns),
        'mean_pos_error': np.mean(all_pos_errors),
        'mean_final_error': np.mean(all_final_pos_errors),
        'returns': all_returns,
        'pos_errors': all_pos_errors,
        'trajectories': all_trajectories,
    }


# Controller functions
def ppo_controller(obs, env):
    action, _, _ = ppo.select_action(obs)
    return np.clip(action, -1, 1)


def sac_controller(obs, env):
    return sac.select_action(obs, deterministic=True)


def pid_controller(obs, env):
    pid = PIDController()
    params = env.params
    thrusts = pid.compute(env.state, env.target_pos, params=params, dt=env.dt)
    action = 2.0 * thrusts / params.max_thrust_per_motor - 1.0
    return np.clip(action, -1, 1)


# Evaluate
env_eval = DroneHoverEnv(target_pos=np.array([0., 0., 1.]), max_steps=200, dt=0.02, seed=99)

print("Evaluating PPO..."); t0 = time.time()
results_ppo = evaluate_controller(ppo_controller, env_eval)
print(f"  Done in {time.time()-t0:.1f}s")

print("Evaluating SAC...")
t0 = time.time()
results_sac = evaluate_controller(sac_controller, env_eval)
print(f"  Done in {time.time()-t0:.1f}s")

print("Evaluating PID...")
t0 = time.time()
results_pid = evaluate_controller(pid_controller, env_eval)
print(f"  Done in {time.time()-t0:.1f}s")

# Robustness test: perturb mass by 20%
env_perturbed = DroneHoverEnv(target_pos=np.array([0., 0., 1.]), max_steps=200, dt=0.02, seed=99)
env_perturbed.params.mass *= 1.2

print("\nRobustness test (mass +20%):")
results_ppo_p = evaluate_controller(ppo_controller, env_perturbed)
results_sac_p = evaluate_controller(sac_controller, env_perturbed)
results_pid_p = evaluate_controller(pid_controller, env_perturbed)

# Print table
print(f"\n{'Metric':<25} {'PPO':>10} {'SAC':>10} {'PID':>10}")
print("-" * 57)
print(f"{'Mean return':<25} {results_ppo['mean_return']:>10.2f} {results_sac['mean_return']:>10.2f} {results_pid['mean_return']:>10.2f}")
print(f"{'Mean pos error [m]':<25} {results_ppo['mean_pos_error']:>10.3f} {results_sac['mean_pos_error']:>10.3f} {results_pid['mean_pos_error']:>10.3f}")
print(f"{'Final pos error [m]':<25} {results_ppo['mean_final_error']:>10.3f} {results_sac['mean_final_error']:>10.3f} {results_pid['mean_final_error']:>10.3f}")
print(f"{'Perturbed return':<25} {results_ppo_p['mean_return']:>10.2f} {results_sac_p['mean_return']:>10.2f} {results_pid_p['mean_return']:>10.2f}")
print(f"{'Perturbed pos err [m]':<25} {results_ppo_p['mean_pos_error']:>10.3f} {results_sac_p['mean_pos_error']:>10.3f} {results_pid_p['mean_pos_error']:>10.3f}")

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
labels = ['PPO', 'SAC', 'PID']
colors = ['#2196F3', '#FF9800', '#4CAF50']

# Returns
ax = axes[0]
normal_returns = [results_ppo['mean_return'], results_sac['mean_return'], results_pid['mean_return']]
perturbed_returns = [results_ppo_p['mean_return'], results_sac_p['mean_return'], results_pid_p['mean_return']]
x = np.arange(len(labels))
width = 0.35
ax.bar(x - width/2, normal_returns, width, label='Normal', color=colors, alpha=0.8)
ax.bar(x + width/2, perturbed_returns, width, label='Mass +20%', color=colors, alpha=0.4)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Mean Return'); ax.set_title('Episode Returns')
ax.legend()

# Position error
ax = axes[1]
normal_err = [results_ppo['mean_pos_error'], results_sac['mean_pos_error'], results_pid['mean_pos_error']]
perturbed_err = [results_ppo_p['mean_pos_error'], results_sac_p['mean_pos_error'], results_pid_p['mean_pos_error']]
ax.bar(x - width/2, normal_err, width, label='Normal', color=colors, alpha=0.8)
ax.bar(x + width/2, perturbed_err, width, label='Mass +20%', color=colors, alpha=0.4)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Mean Pos Error [m]'); ax.set_title('Tracking Error')
ax.legend()

# Robustness delta
ax = axes[2]
delta_returns = [n - p for n, p in zip(normal_returns, perturbed_returns)]
ax.bar(labels, delta_returns, color=colors, alpha=0.8)
ax.set_ylabel('Return Drop (Normal - Perturbed)')
ax.set_title('Robustness: Return Degradation')
ax.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

# -- 3D Trajectory Comparison: PPO vs SAC vs PID (time-colored) --
fig = plt.figure(figsize=(18, 12))
all_results = [
    ('PPO', results_ppo, '#1f77b4'),
    ('SAC', results_sac, '#ff7f0e'),
    ('PID', results_pid, '#2ca02c'),
]
target = np.array([0.0, 0.0, 1.0])

# ── Row 1: Time-colored 3D trajectories ─────────────────────────────
for idx, (name, res, color) in enumerate(all_results):
    ax = fig.add_subplot(2, 3, idx + 1, projection='3d')
    for traj in res.get('trajectories', []):
        if len(traj) > 1:
            t_norm = np.linspace(0, 1, len(traj))
            colors_t = plt.cm.coolwarm(t_norm)
            for j in range(len(traj) - 1):
                ax.plot(traj[j:j+2, 0], traj[j:j+2, 1], traj[j:j+2, 2],
                        color=colors_t[j], lw=1.5, alpha=0.8)
            ax.scatter(*traj[0], c='blue', s=50, marker='o', zorder=5,
                       edgecolors='k', linewidths=0.5, label='Start')
            ax.scatter(*traj[-1], c='red', s=50, marker='s', zorder=5,
                       edgecolors='k', linewidths=0.5, label='End')
    ax.scatter(*target, c='gold', s=250, marker='*',
               edgecolors='k', linewidths=1.2, zorder=10, label='Target')
    # Settling time annotation: find when error < 0.3m
    best_traj = res['trajectories'][0] if res.get('trajectories') else None
    if best_traj is not None and len(best_traj) > 1:
        errors_t = np.linalg.norm(best_traj - target, axis=1)
        settled_idx = np.where(errors_t < 0.3)[0]
        if len(settled_idx) > 0:
            t_settle = settled_idx[0] * 0.02
            ax.set_title(f'{name}  (err={res["mean_pos_error"]:.3f}m)\n'
                         f'settling ≈ {t_settle:.2f}s', fontsize=10, fontweight='bold')
        else:
            ax.set_title(f'{name}  (err={res["mean_pos_error"]:.3f}m)\n'
                         f'not settled', fontsize=10, fontweight='bold')
    else:
        ax.set_title(f'{name}\nerr={res["mean_pos_error"]:.3f}m', fontsize=10)
    ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]'); ax.set_zlabel('Z [m]')
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_zlim(-1, 3)
    ax.legend(fontsize=6, loc='upper right')

# ── Row 2: Per-axis position error over time ─────────────────────────
axis_labels = ['X', 'Y', 'Z']
axis_colors = ['#e74c3c', '#2ecc71', '#3498db']
for idx, (name, res, _) in enumerate(all_results):
    ax = fig.add_subplot(2, 3, idx + 4)
    best_traj = res['trajectories'][0] if res.get('trajectories') else None
    if best_traj is not None and len(best_traj) > 1:
        t_axis = np.arange(len(best_traj)) * 0.02
        for dim in range(3):
            error_dim = best_traj[:, dim] - target[dim]
            ax.plot(t_axis, error_dim, color=axis_colors[dim],
                    lw=1.8, alpha=0.85, label=f'{axis_labels[dim]} error')
        ax.axhline(0, color='black', lw=0.5, alpha=0.5)
        ax.fill_between(t_axis, -0.1, 0.1, alpha=0.06, color='green',
                        label='±0.1m band')
    ax.set_xlabel('Time [s]'); ax.set_ylabel('Position Error [m]')
    ax.set_title(f'{name}: Per-Axis Error', fontweight='bold', fontsize=10)
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

plt.suptitle('Controller Flight Paths — Time-Colored Trajectories (blue→red)\n'
             '5 eval episodes per controller',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Domain Randomization

**Sim-to-real transfer** is the central challenge for deploying RL policies
on real drones. The simulation never perfectly matches reality (*reality gap*).

Domain randomization (Tobin et al., 2017; Hwangbo et al., 2019) addresses
this by training the policy across a *distribution* of environments:

$$\theta^* = \arg\max_\theta \; \mathbb{E}_{\xi \sim p(\xi)}\left[
J(\theta; \xi)\right]$$

where $\xi$ encodes randomized parameters:
- **Mass**: $m \sim \mathcal{U}(0.8 m_0, 1.2 m_0)$
- **Inertia**: $I_{xx}, I_{yy}, I_{zz} \sim \mathcal{U}(0.8 I_0, 1.2 I_0)$
- **Motor gains**: $k_f \sim \mathcal{U}(0.9 k_{f,0}, 1.1 k_{f,0})$
- **Action latency**: $\tau \sim \mathcal{U}(0, 20\text{ms})$

### Intuition

If the policy achieves good performance across *all* plausible parameter
variations in simulation, the real-world parameters are (with high probability)
within this distribution, and the policy will transfer.

### Five Key Factors for Zero-Shot Sim-to-Real (SimpleFlight, 2025)

The SimpleFlight study (Xu et al., 2025) identifies the **five most impactful factors**
for zero-shot transfer of PPO policies to real Crazyflie quadrotors, reducing tracking
error by >50% vs baselines:

1. **Rotation matrix in observations** (not Euler angles) — avoids gimbal lock
   and provides the actor with a representation that is smooth under composition
2. **Time vector in critic input** — the critic receives $(t, T-t)$ as input,
   which helps the value function account for time-varying difficulty in trajectory tracking
3. **Action difference regularization** — reward penalty $-\alpha \| a_t - a_{t-1} \|^2$
   produces smooth motor commands that physical actuators can track without oscillation
4. **System identification + selective randomization** — identify the true mass,
   inertia, and motor constants on hardware first; only randomize parameters that
   cannot be identified precisely (latency, aerodynamic effects)
5. **Large batch sizes** — PPO benefits from large batches (>8192) for stable gradient
   estimates; this dominates over learning rate tuning

### Action Representations for Quadrotors

| Action Space | Description | Sim-to-Real Quality |
|:------------|:-----------|:-------------------|
| **Motor RPMs** | Direct motor commands | Poor — requires accurate motor model |
| **Body rates** | Roll/pitch/yaw rates + thrust | Good — low-level PID handles motors |
| **CTBR** | Collective thrust + body rates | **Best** — matches onboard PID interface |

**CTBR** (Collective Thrust + Body Rate) is the recommended action space because
it matches the interface that real flight controllers (PX4, Crazyflie) expose,
minimising the sim-to-real gap at the actuator level.

### RAPTOR: Foundation Policies for Quadrotor Control (2025)

RAPTOR (Bauersfeld et al., 2025) demonstrates that a **single 2084-parameter** recurrent
policy can control 10+ different real quadrotors (32g–2.4kg) with zero-shot transfer:

- **Meta-Imitation Learning**: train 1000 SAC teacher policies (one per randomised
  quadrotor) → distil into one adaptive student with a recurrent hidden layer
- The recurrence enables **implicit system identification**: within milliseconds of
  interaction, the hidden state adapts to the specific platform's dynamics
- Key finding: off-policy SAC produces more stable teacher policies than PPO for
  this distillation setting

This represents the shift from *per-platform* RL (SimpleFlight) to *platform-agnostic*
foundation policies — analogous to how foundation models work in vision and language.

### E2E-Fly: End-to-End Visual Flight Control (2026)

E2E-Fly (Liu et al., 2026) pushes RL for quadrotors to the **vision-in, action-out**
regime: a single policy maps raw RGB (or depth) images directly to CTBR
(collective thrust + body rates) without an explicit state estimator or planner.

| Component | Design choice | Why it matters |
|-----------|--------------|----------------|
| Observation | Stacked RGB frames (or depth) | Captures motion cues without optical flow |
| Action | CTBR (not motor RPM) | Decouples policy from platform-specific mixing |
| Training | PPO + domain randomization | Same algorithm we implement in `src/rl_agents.py` |
| Sim | Isaac Gym / custom physics | Millions of parallel rollouts for sample efficiency |
| Real transfer | CTBR + delay-aware sim | Matches REX and SimpleFlight findings |

The key insight for our workshop: **E2E-Fly does not replace the Perceive-Map-Plan-Act
pipeline** (Notebook 24) — it *collapses* it into a single learned function. This works
for reactive flight in structured environments but lacks the safety guarantees of
explicit mapping and planning. Production systems (Skydio, DJI) use the modular pipeline;
research systems (E2E-Fly, RAPTOR) push the end-to-end frontier.

### Delay-Aware Training (REX Framework, Delft 2025)

A subtle but critical factor: **communication and computation latency** between
sensing and actuation can destabilise RL policies trained in zero-latency simulation.
The REX framework demonstrates that:

- Policies trained **with** delay simulation maintain stable flight in real-world
  deployment; policies trained **without** delay oscillate or crash
- The effect is most pronounced at aggressive manoeuvre speeds and small radii
- Delay-aware simulation is cheap to implement: buffer the action queue and apply
  actions $k$ steps later, where $k = \lceil \tau_{\text{latency}} / \Delta t \rceil$

### Mathematical Foundations of Domain Randomization

Why does training across randomised parameters produce policies that transfer
to the real world? Two complementary perspectives explain this:

#### Perspective 1: Robust Optimisation

Domain randomization solves a **distributionally robust** optimisation problem.
Let $\xi$ be the vector of uncertain physical parameters (mass, inertia,
motor gains, latency) and let $\Xi$ be the set of plausible values. The
DR objective is:

$$\theta^* = \arg\max_\theta \; \mathbb{E}_{\xi \sim \mathcal{U}(\Xi)}\!\left[J(\theta; \xi)\right]$$

If the real-world parameters $\xi_{\text{real}} \in \Xi$, then $J(\theta^*; \xi_{\text{real}})$
is guaranteed to be at least as good as the *worst-case* performance over
$\Xi$. More precisely, for any confidence level $\delta$:

$$\Pr_{\xi \sim \mathcal{U}(\Xi)}\!\left[J(\theta^*; \xi) \ge J_{\min}\right] \ge 1 - \delta$$

The key design choice is the **width** of $\Xi$: too narrow and the real
parameters may lie outside; too wide and the policy becomes overly
conservative (it must handle wildly different dynamics simultaneously).

#### Perspective 2: Implicit System Identification

An alternative view: a policy trained under DR learns to be **adaptive**.
Because the dynamics change every episode, the policy must infer the
current parameter regime from recent observations. This is equivalent
to performing online system identification *implicitly* through the
policy's hidden representation.

This perspective explains why **recurrent policies** (LSTM, GRU) and
**transformer-based policies** are particularly effective with DR: their
hidden state naturally accumulates information about the current dynamics,
enabling rapid adaptation within an episode.

The RAPTOR framework (Bauersfeld et al., 2025) formalises this: a recurrent
policy with a 64-dimensional hidden state adapts to a new quadrotor
platform within $\sim$50ms of flight ($\sim$3 control steps at 50 Hz).

#### Practical Guidelines for DR Ranges

The SimpleFlight study provides empirically validated DR ranges for
Crazyflie-class quadrotors:

| Parameter | Nominal | DR Range | Sensitivity |
|:----------|:--------|:---------|:------------|
| Mass | 0.033 kg | ±20% | High — directly affects hover thrust |
| Inertia ($I_{xx}$) | $1.4 \times 10^{-5}$ kg·m² | ±20% | Medium — affects angular response |
| Motor time constant | 0.04 s | ±50% | **Very high** — most impactful factor |
| Motor thrust coefficient | varies | ±10% | Medium |
| Drag coefficient | varies | ±30% | Low for hover, high for fast flight |
| Action latency | 0 ms | 0–20 ms | **Very high** — can destabilise if unmodelled |
| IMU noise (gyro) | 0 | $\sigma \in [0, 0.02]$ rad/s | Medium |

> **Rule of thumb** (SimpleFlight): randomise latency and motor time
> constants *first* — these two parameters account for >60% of the
> sim-to-real gap for hover and tracking tasks. Mass and inertia
> randomisation provides diminishing returns once actuator dynamics
> are well-modelled.

#### Adaptive Domain Randomization (ADR)

Fixed uniform randomization can be improved with **ADR** (OpenAI, 2019):
the randomization ranges are *expanded* automatically when the policy
achieves a success threshold, and *contracted* when it fails. This
curriculum over randomization parameters produces more robust policies
than fixed-range DR:

$$\Xi_{k+1} = \begin{cases}
\text{expand}(\Xi_k) & \text{if success rate} > \tau_{\text{high}} \\
\text{contract}(\Xi_k) & \text{if success rate} < \tau_{\text{low}} \\
\Xi_k & \text{otherwise}
\end{cases}$$

ADR was critical for the OpenAI Rubik's cube result and has since been
adopted for quadrotor sim-to-real by several groups (including RAPTOR's
teacher training pipeline).

In [ ]:
# ── Domain Randomization for robust hover control ───────────────────

class DomainRandomizedHoverEnv:
    """Wrapper that randomizes physical parameters each reset."""

    def __init__(self, base_seed=42, randomize=True):
        self.base_env = DroneHoverEnv(
            target_pos=np.array([0., 0., 1.]),
            max_steps=200, dt=0.02, seed=base_seed
        )
        self.randomize = randomize
        self.rng = np.random.RandomState(base_seed)
        self.obs_dim = self.base_env.obs_dim
        self.act_dim = self.base_env.act_dim
        self.nominal_mass = self.base_env.params.mass
        self.nominal_I = self.base_env.params.I.copy()
        self.nominal_kf = self.base_env.params.k_f

    def reset(self):
        if self.randomize:
            mass_scale = self.rng.uniform(0.8, 1.2)
            self.base_env.params.mass = self.nominal_mass * mass_scale

            inertia_scale = self.rng.uniform(0.8, 1.2, size=3)
            for i in range(3):
                self.base_env.params.I[i, i] = self.nominal_I[i, i] * inertia_scale[i]
                self.base_env.params.I_inv[i, i] = 1.0 / self.base_env.params.I[i, i]

            kf_scale = self.rng.uniform(0.9, 1.1)
            self.base_env.params.k_f = self.nominal_kf * kf_scale

        return self.base_env.reset()

    def step(self, action):
        return self.base_env.step(action)

    @property
    def state(self):
        return self.base_env.state

    @property
    def params(self):
        return self.base_env.params

    @property
    def target_pos(self):
        return self.base_env.target_pos


def train_ppo_env(env, n_episodes=40, seed=42):
    agent = PPO(obs_dim=env.obs_dim, act_dim=env.act_dim,
                lr_policy=5e-4, lr_value=1e-3, gamma=0.99, lam=0.95,
                clip_eps=0.2, n_epochs=3, batch_size=64, seed=seed)
    returns_hist = []
    for ep in range(n_episodes):
        obs = env.reset()
        ep_ret = 0.0
        for step in range(200):
            action, log_prob, value = agent.select_action(obs)
            action_clipped = np.clip(action, -1, 1)
            next_obs, reward, done, _ = env.step(action_clipped)
            agent.store_transition(obs, action_clipped, reward, done, log_prob, value)
            ep_ret += reward
            obs = next_obs
            if done:
                obs = env.reset()
        agent.update(obs)
        returns_hist.append(ep_ret)
    return agent, returns_hist


# Train with and without domain randomization
print("Training with domain randomization..."); t0 = time.time()
env_dr = DomainRandomizedHoverEnv(base_seed=42, randomize=True)
agent_dr, returns_dr = train_ppo_env(env_dr, n_episodes=40)
print(f"  Done in {time.time()-t0:.1f}s")

print("Training without domain randomization...")
t0 = time.time()
env_fixed = DomainRandomizedHoverEnv(base_seed=42, randomize=False)
agent_fixed, returns_fixed = train_ppo_env(env_fixed, n_episodes=40)
print(f"  Done in {time.time()-t0:.1f}s")

# Test robustness: sweep mass from 0.7x to 1.4x
mass_scales = np.linspace(0.7, 1.4, 15)
dr_errors = []
fixed_errors = []

for ms in mass_scales:
    test_env = DroneHoverEnv(target_pos=np.array([0., 0., 1.]),
                             max_steps=200, dt=0.02, seed=99)
    test_env.params.mass = 0.5 * ms

    # DR agent
    obs = test_env.reset()
    errs_dr = []
    for _ in range(200):
        action, _, _ = agent_dr.select_action(obs)
        obs, _, done, _ = test_env.step(np.clip(action, -1, 1))
        errs_dr.append(np.linalg.norm(obs[:3]))
        if done:
            break
    dr_errors.append(np.mean(errs_dr))

    # Fixed agent
    obs = test_env.reset()
    errs_fx = []
    for _ in range(200):
        action, _, _ = agent_fixed.select_action(obs)
        obs, _, done, _ = test_env.step(np.clip(action, -1, 1))
        errs_fx.append(np.linalg.norm(obs[:3]))
        if done:
            break
    fixed_errors.append(np.mean(errs_fx))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(returns_dr, label='With DR', alpha=0.8)
ax.plot(returns_fixed, label='Without DR', alpha=0.8)
ax.set_xlabel('Episode'); ax.set_ylabel('Return')
ax.set_title('Training Curves: Domain Randomization')
ax.legend()

ax = axes[1]
ax.plot(mass_scales, dr_errors, 'o-', label='With DR', alpha=0.8)
ax.plot(mass_scales, fixed_errors, 's-', label='Without DR', alpha=0.8)
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5, label='Nominal mass')
ax.fill_betweenx([0, max(max(dr_errors), max(fixed_errors)) * 1.1],
                  0.8, 1.2, alpha=0.1, color='green', label='DR range')
ax.set_xlabel('Mass Scale Factor')
ax.set_ylabel('Mean Position Error [m]')
ax.set_title('Robustness: Mass Variation')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nMean error across all mass scales:")
print(f"  With DR:    {np.mean(dr_errors):.3f} m")
print(f"  Without DR: {np.mean(fixed_errors):.3f} m")

## 10. Curriculum Learning

Curriculum learning (Bengio et al., 2009) presents tasks in order of
increasing difficulty — analogous to how a flight student starts with
simulator exercises before progressing to real aircraft.

### The Curriculum for Drone Control

A typical training curriculum proceeds through four phases of increasing
difficulty:

| Phase | Task | Difficulty | What the Agent Learns |
|:------|:-----|:-----------|:---------------------|
| 1 | Hover near target | $d_0 = 0.05$ m displacement | Basic attitude stabilisation |
| 2 | Hover from far away | $d = 1.0$ m displacement | Large-scale position control |
| 3 | Track moving waypoints | Time-varying $p_d(t)$ | Velocity control, anticipation |
| 4 | Navigate with obstacles + wind | Full task complexity | Obstacle avoidance, robustness |

### Why Curriculum Learning is Essential for Drones

The intuition is straightforward: **early rewards from easy tasks provide
a strong learning signal**. As the policy improves, harder tasks become
feasible. Without a curriculum, the initial policy may never receive useful
reward signal on the hard task.

More formally, curriculum learning addresses the **exploration problem**.
Consider a hover task with initial displacement $d = 1.0$ m. A random
policy generates roughly:

$$r_{\text{random}} \approx -w_p d^2 \cdot T = -1.0 \cdot 200 = -200$$

This is deep in the low-reward region — the gradient signal is dominated
by noise. With $d = 0.05$ m:

$$r_{\text{random}} \approx -w_p (0.05)^2 \cdot T = -0.5$$

Now the agent is close enough to the high-reward region that even random
perturbations occasionally produce positive feedback. The policy gradient
has a clear signal to follow.

### The Difficulty Schedule

We progressively increase the initial displacement from the target
during training. The difficulty parameter $d_k$ at episode $k$ follows
a linear ramp:

$$d_k = d_{\min} + (d_{\max} - d_{\min}) \cdot \min\!\left(1, \frac{k}{K}\right)$$

where $K$ is the number of episodes to reach full difficulty. In our
implementation, $K = 0.6 \cdot N$ (60% of total training).

**Alternative schedules** used in practice:

- **Performance-based**: advance difficulty only when the current level
  is "solved" (success rate $> 80\%$). More robust but requires defining
  success criteria.
- **Exponential ramp**: $d_k = d_{\min} \cdot (d_{\max}/d_{\min})^{k/K}$.
  Spends more time on easy tasks, which can be better for very hard final
  tasks.
- **Automatic Curriculum Learning (ACL)**: a meta-learner selects the
  difficulty that maximises the *learning progress* (change in success
  rate), not the current success rate. This avoids wasting time on
  already-mastered difficulties.

### Connection to Reward Shaping

Curriculum learning and reward shaping are complementary strategies for
the same problem (sparse reward signals):

- **Reward shaping** makes the reward landscape smoother → the agent
  receives gradient everywhere
- **Curriculum learning** starts the agent in easy states → the agent
  is already near good reward regions

Using *both* together is standard practice in state-of-the-art drone RL
systems (SimpleFlight, RAPTOR, E2E-Fly). The combination can reduce
training time by 5–10$\times$ compared to either technique alone.

In [ ]:
# ── Curriculum learning: vary initial displacement ──────────────────

class CurriculumHoverEnv:
    """Hover env with controllable initial displacement."""

    def __init__(self, max_displacement=0.3, seed=42):
        self.base_env = DroneHoverEnv(
            target_pos=np.array([0., 0., 1.]),
            max_steps=200, dt=0.02, seed=seed
        )
        self.obs_dim = self.base_env.obs_dim
        self.act_dim = self.base_env.act_dim
        self.displacement = max_displacement
        self.rng = np.random.RandomState(seed)

    def set_difficulty(self, displacement):
        self.displacement = displacement

    def reset(self):
        from src.drone import QuadrotorState
        self.base_env.state = QuadrotorState()
        self.base_env.state.position = (
            self.base_env.target_pos + self.rng.randn(3) * self.displacement
        )
        self.base_env.state.velocity = self.rng.randn(3) * 0.05 * self.displacement
        self.base_env.step_count = 0
        return self.base_env._get_obs()

    def step(self, action):
        return self.base_env.step(action)

    @property
    def state(self):
        return self.base_env.state

    @property
    def target_pos(self):
        return self.base_env.target_pos


def train_with_curriculum(use_curriculum=True, n_episodes=50, seed=42):
    env = CurriculumHoverEnv(seed=seed)
    agent = PPO(obs_dim=env.obs_dim, act_dim=env.act_dim,
                lr_policy=5e-4, lr_value=1e-3, gamma=0.99, lam=0.95,
                clip_eps=0.2, n_epochs=3, batch_size=64, seed=seed)

    d_min, d_max = 0.05, 1.0
    curriculum_episodes = int(0.6 * n_episodes)
    returns_hist = []
    difficulty_hist = []

    for ep in range(n_episodes):
        if use_curriculum:
            d = d_min + (d_max - d_min) * min(1.0, ep / curriculum_episodes)
        else:
            d = d_max
        env.set_difficulty(d)
        difficulty_hist.append(d)

        obs = env.reset()
        ep_ret = 0.0
        for step in range(200):
            action, log_prob, value = agent.select_action(obs)
            action_clipped = np.clip(action, -1, 1)
            next_obs, reward, done, _ = env.step(action_clipped)
            agent.store_transition(obs, action_clipped, reward, done, log_prob, value)
            ep_ret += reward
            obs = next_obs
            if done:
                obs = env.reset()
        agent.update(obs)
        returns_hist.append(ep_ret)

    return returns_hist, difficulty_hist


print("Training WITH curriculum..."); t0 = time.time()
ret_curriculum, diff_hist = train_with_curriculum(use_curriculum=True, n_episodes=50)
print(f"  Done in {time.time()-t0:.1f}s")

print("Training WITHOUT curriculum...")
t0 = time.time()
ret_no_curriculum, _ = train_with_curriculum(use_curriculum=False, n_episodes=50)
print(f"  Done in {time.time()-t0:.1f}s")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ax = axes[0]
ax.plot(diff_hist, linewidth=2)
ax.set_xlabel('Episode'); ax.set_ylabel('Initial Displacement [m]')
ax.set_title('Curriculum: Difficulty Schedule')

ax = axes[1]
ax.plot(ret_curriculum, label='With Curriculum', alpha=0.7)
ax.plot(ret_no_curriculum, label='Without Curriculum', alpha=0.7)
ax.set_xlabel('Episode'); ax.set_ylabel('Return')
ax.set_title('Raw Learning Curves')
ax.legend()

ax = axes[2]
window = 5
c_smooth = np.convolve(ret_curriculum, np.ones(window)/window, mode='valid')
nc_smooth = np.convolve(ret_no_curriculum, np.ones(window)/window, mode='valid')
ax.plot(c_smooth, label='With Curriculum', alpha=0.8, linewidth=2)
ax.plot(nc_smooth, label='Without Curriculum', alpha=0.8, linewidth=2)
ax.set_xlabel('Episode'); ax.set_ylabel('Smoothed Return')
ax.set_title('Smoothed Learning Curves')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nFinal avg return (last 10 episodes):")
print(f"  With curriculum:    {np.mean(ret_curriculum[-10:]):8.2f}")
print(f"  Without curriculum: {np.mean(ret_no_curriculum[-10:]):8.2f}")

## The 2026 Landscape: What Actually Works for Real Drone Deployment

The gap between RL-in-simulation and RL-on-real-drones remains the field's
central challenge. Here we summarize the state of the art as of 2026, drawing
on recent surveys and deployment reports.

### Three Paradigms for Vision-Based Drone Learning

Zhang et al. (*IEEE TNNLS*, 2025) categorise learned drone control into
three paradigms of increasing ambition:

| Paradigm | What RL Controls | What Remains Classical | Deployment Status |
|:---------|:----------------|:----------------------|:-----------------|
| **Indirect** | High-level decisions (waypoints, speeds) | Geometric / PID inner loop | Production-ready |
| **Semidirect** | Reference signals (body rates, thrust setpoints) | Low-level rate controller | Demonstrated on hardware |
| **End-to-end** | Pixels $\to$ motor commands | Nothing | Research prototypes only |

**The consensus**: hybrid architectures — RL for high-level decisions, geometric
control for low-level actuation — dominate real-world deployment. End-to-end
approaches (E2E-Fly, Liu et al., 2026) achieve impressive results in structured
simulation but lack the safety guarantees required by aviation regulators.

### SimpleFlight: What Actually Matters for Zero-Shot Sim-to-Real (arXiv 2412.11764, 2025)

The most rigorous ablation study on sim-to-real RL for quadrotors identifies
**five critical factors** — most prior work tuned these haphazardly:

| Factor | Design Choice | Why It Matters |
|--------|---------------|----------------|
| **Actor input** | Velocity $\mathbf{v}$ + full rotation matrix $R$ | Rotation matrix avoids gimbal lock and sign ambiguities of Euler/quaternion |
| **Critic input** | Add a *time vector* encoding episode progress | Helps value function model time-varying cost landscape |
| **Reward shaping** | Action *difference* penalty $\|\Delta a\|^2$ | Smooths motor commands; prevents oscillation on real hardware |
| **System ID** | Measure real motor lag, mass, CoG; randomise *only* uncertain params | Over-randomisation wastes capacity; targeted randomisation transfers |
| **Training** | Large batch sizes ($\geq 65536$) | PPO's surrogate objective is high-variance with small batches |

**Result on Crazyflie hardware:** SimpleFlight reduces trajectory tracking error by
**>50%** compared to SOTA RL baselines (including the RAPTOR framework), with zero
fine-tuning on the real platform.

**Key lesson for this notebook:** Our PPO implementation covers the core algorithm,
but real deployment requires the engineering details above. The gap between "PPO works
in simulation" and "PPO flies a real drone" is primarily a *systems* gap, not an
*algorithm* gap.

### Deployment Readiness Survey (Machines, 2025; Spectrum of Engineering Sciences, 2026)

Two systematic surveys (187 papers) establish the current quantitative picture:

- **PPO and SAC** are >2× more effective than value-based methods (DQN, DDQN) for
  continuous UAV control — consistent across navigation, hovering, and landing tasks
- RL-based methods achieve **20–45% improvement** in control accuracy and **30–60%
  reduction** in collision frequency vs. classical controllers — *in simulation*
- **Sim-to-real transfer** degrades performance by ~35% without robustness-oriented
  training (domain randomization, system identification, action smoothing)
- Safety-constrained RL (Lagrangian relaxation, shielding) reduces violations but
  introduces **10–20% training efficiency** overhead

### AceFormer: Active Exploration Without Expert Demonstrations (2025)

**AceFormer** introduces a *transformer-based* active exploration policy for GNSS-denied
navigation:

- Learns to actively explore unknown environments without any expert demonstrations
- Uses a transformer architecture to maintain a belief state over the map
- Generalises across environment geometries not seen during training
- Achieves state-of-the-art on active SLAM benchmarks

### Multi-Agent RL (MARL) for Drone Swarms

Coordinating multiple drones is an emerging frontier:

- **Centralised training, decentralised execution (CTDE)**: each drone
  runs its own policy network at inference time, but all policies are
  trained jointly with a shared critic that sees the full swarm state
- **Communication-constrained MARL**: agents learn *what* to communicate
  (compressed state summaries) alongside *how* to act
- **Current limitations**: MARL policies are brittle to agent dropout,
  communication latency, and heterogeneous platforms; robust deployment
  remains several years away
- The 2026 Atlantis Press review notes that most MARL-for-drones papers
  evaluate in simplified 2D environments — transfer to 3D with realistic
  aerodynamics is largely unexplored

### Key Simulators for RL-Based Drone Research

| Simulator | Status | Strengths | Limitations |
|:----------|:-------|:----------|:------------|
| **AirSim** (Microsoft) | Deprecated (2022) | Realistic rendering, PX4 integration | No longer maintained |
| **Isaac Sim + Orbit** (NVIDIA) | Active | GPU-parallelised physics, thousands of drones | Heavy GPU requirements |
| **Flightmare 2.0** (UZH) | Active | Fast rendering, flexible dynamics | Smaller community |
| **PyBullet / pybullet-drones** | Active | Lightweight, easy to modify | Limited visual fidelity |
| **gym-pybullet-drones** | Active | OpenAI Gym API, multi-agent | Simplified aerodynamics |

**Recommendation for this workshop**: `gym-pybullet-drones` is the easiest
on-ramp for RL experimentation. For production sim-to-real pipelines, NVIDIA
Isaac Sim + Orbit offers the best domain randomization and parallelism.

### The Road Ahead

The field is converging on a **three-layer stack** for autonomous drone deployment:

1. **Foundation policy layer** (RAPTOR-style): a single adaptive policy that
   works across platforms via implicit system identification
2. **Task-specific fine-tuning**: curriculum learning + domain randomization
   for the target mission (inspection, delivery, search-and-rescue)
3. **Safety monitor**: a classical controller or learned backup policy that
   intervenes when the RL policy's uncertainty exceeds a threshold

This mirrors the evolution in autonomous driving: end-to-end is the research
frontier, but production systems layer learned components onto verified
classical backbones.

## Exercises

### Exercise 1: Implement TD3 (Twin Delayed DDPG)

TD3 (Fujimoto et al., ICML 2018) improves DDPG with three tricks:

1. **Clipped Double-Q**: Use two Q-networks and take the minimum for targets:
   $$y = r + \gamma \min_{i=1,2} Q_{\phi_i'}(s', \tilde{a}')$$

2. **Delayed Policy Updates**: Update the policy and target networks every $d$ steps
   (typically $d = 2$) rather than every step.

3. **Target Policy Smoothing**: Add noise to target actions:
   $$\tilde{a}' = \text{clip}\!\left(\mu_{\theta'}(s') + \text{clip}(\epsilon, -c, c),\; a_{\text{low}},\; a_{\text{high}}\right), \quad \epsilon \sim \mathcal{N}(0, \sigma)$$

### Exercise 2: Automatic entropy tuning for SAC

Instead of a fixed $\alpha$, solve the constrained optimization:

$$\alpha^* = \arg\min_\alpha \; \mathbb{E}_{a \sim \pi^*}\left[
-\alpha \log \pi^*(a|s) - \alpha \bar{\mathcal{H}}
\right]$$

where $\bar{\mathcal{H}} = -\dim(\mathcal{A})$ is the target entropy.

### Exercise 3: Multi-waypoint navigation

Extend the navigation environment to visit a sequence of waypoints.
Design a reward function that incentivizes visiting them in order.

In [ ]:
# ── TD3 (Twin Delayed DDPG) ───────────────────────────────────────────
#
# TD3 combines DDPG with three stabilization tricks.
# Fill in the missing pieces marked with ???.

class TD3:
    """Twin Delayed DDPG (Fujimoto et al., ICML 2018)."""

    def __init__(
        self,
        obs_dim: int,
        act_dim: int,
        gamma: float = 0.99,
        tau: float = 0.005,
        policy_noise: float = 0.2,
        noise_clip: float = 0.5,
        policy_delay: int = 2,
        lr: float = 3e-4,
        buffer_size: int = 100000,
        batch_size: int = 256,
        seed: int = 42,
    ):
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.gamma = gamma
        self.tau = tau
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.policy_delay = policy_delay
        self.lr = lr
        self.batch_size = batch_size

        # Deterministic policy: maps obs -> action
        self.policy = MLPPolicy(obs_dim, act_dim, seed=seed)

        # Twin Q-networks
        self.q1 = MLPValueFunction(obs_dim + act_dim, seed=seed + 1)
        self.q2 = MLPValueFunction(obs_dim + act_dim, seed=seed + 2)

        # Target networks (initialized as copies)
        self.q1_target = MLPValueFunction(obs_dim + act_dim, seed=seed + 1)
        self.q2_target = MLPValueFunction(obs_dim + act_dim, seed=seed + 2)
        self.policy_target = MLPPolicy(obs_dim, act_dim, seed=seed)

        self.buffer = ReplayBuffer(buffer_size, obs_dim, act_dim)
        self.rng = np.random.RandomState(seed)
        self.update_count = 0

    def select_action(self, obs, noise_scale=0.1):
        """Select action using deterministic policy + exploration noise."""
        mean, _ = self.policy.forward(obs)
        noise = self.rng.randn(*mean.shape) * noise_scale
        return np.clip(mean + noise, -1.0, 1.0)

    def store_transition(self, obs, action, reward, next_obs, done):
        self.buffer.add(obs, action, reward, next_obs, done)

    def update(self):
        """TD3 update step."""
        if self.buffer.size < self.batch_size:
            return {}

        self.update_count += 1
        batch = self.buffer.sample(self.batch_size, self.rng)
        obs = batch["obs"]
        actions = batch["actions"]
        rewards = batch["rewards"]
        next_obs = batch["next_obs"]
        dones = batch["dones"]

        # ── Step 1: Target policy smoothing ──────────────────────────
        # Compute target actions with clipped noise:
        #   noise = clip(N(0, policy_noise), -noise_clip, noise_clip)
        #   a' = clip(mu_target(s') + noise, -1, 1)
        next_means, _ = self.policy_target.forward(next_obs)
        target_noise = np.clip(
            self.rng.randn(*next_means.shape) * self.policy_noise,
            -self.noise_clip, self.noise_clip
        )
        next_actions = np.clip(next_means + target_noise, -1.0, 1.0)

        # ── Step 2: Clipped double-Q target ──────────────────────────
        # y = r + gamma * (1 - done) * min(Q1_target, Q2_target)
        sa_next = np.concatenate([next_obs, next_actions], axis=-1)
        q1_targ = self.q1_target.forward(sa_next)
        q2_targ = self.q2_target.forward(sa_next)
        min_q_target = np.minimum(q1_targ, q2_targ)
        target_q = rewards + self.gamma * (1 - dones) * min_q_target

        # ── Step 3: Critic loss ──────────────────────────────────────
        sa = np.concatenate([obs, actions], axis=-1)
        q1_val = self.q1.forward(sa)
        q2_val = self.q2.forward(sa)
        critic_loss = np.mean((q1_val - target_q)**2) + np.mean((q2_val - target_q)**2)

        # ── Step 4: Delayed policy update ────────────────────────────
        # Only update policy every `policy_delay` steps
        policy_loss = 0.0
        if self.update_count % self.policy_delay == 0:
            # Policy loss: -mean(Q1(s, mu(s)))
            curr_means, _ = self.policy.forward(obs)
            sa_policy = np.concatenate([obs, curr_means], axis=-1)
            policy_loss = -np.mean(self.q1.forward(sa_policy))

            # Soft update target networks
            self._soft_update()

        return {
            "critic_loss": float(critic_loss),
            "policy_loss": float(policy_loss),
            "q1_mean": float(np.mean(q1_val)),
        }

    def _soft_update(self):
        """Polyak averaging for all target networks."""
        tau = self.tau
        for w, wt in zip(self.q1.weights, self.q1_target.weights):
            wt[:] = tau * w + (1 - tau) * wt
        for b, bt in zip(self.q1.biases, self.q1_target.biases):
            bt[:] = tau * b + (1 - tau) * bt
        self.q1_target.w_out[:] = tau * self.q1.w_out + (1 - tau) * self.q1_target.w_out
        self.q1_target.b_out[:] = tau * self.q1.b_out + (1 - tau) * self.q1_target.b_out

        for w, wt in zip(self.q2.weights, self.q2_target.weights):
            wt[:] = tau * w + (1 - tau) * wt
        for b, bt in zip(self.q2.biases, self.q2_target.biases):
            bt[:] = tau * b + (1 - tau) * bt
        self.q2_target.w_out[:] = tau * self.q2.w_out + (1 - tau) * self.q2_target.w_out
        self.q2_target.b_out[:] = tau * self.q2.b_out + (1 - tau) * self.q2_target.b_out

        # Update target policy
        src_params = self.policy.get_params()
        tgt_params = self.policy_target.get_params()
        blended = []
        for sp, tp in zip(src_params, tgt_params):
            blended.append(tau * sp + (1 - tau) * tp)
        self.policy_target.set_params(blended)


# Quick test
td3 = TD3(obs_dim=12, act_dim=4, seed=42)
env_td3 = DroneHoverEnv(target_pos=np.array([0., 0., 1.]),
                        max_steps=200, dt=0.02, seed=42)

td3_rewards = []
n_td3_episodes = 30

t0 = time.time()
for ep in range(n_td3_episodes):
    obs = env_td3.reset()
    ep_reward = 0.0
    done = False
    while not done:
        action = td3.select_action(obs, noise_scale=0.1)
        next_obs, reward, done, _ = env_td3.step(action)
        td3.store_transition(obs, action, reward, next_obs, done)
        td3.update()
        ep_reward += reward
        obs = next_obs
    td3_rewards.append(ep_reward)

    if (ep + 1) % 10 == 0:
        print(f"TD3 Episode {ep+1:3d} | Return: {ep_reward:8.2f}")

print(f"\nTD3 training time: {time.time()-t0:.1f}s")
print(f"TD3 final avg return (last 10): {np.mean(td3_rewards[-10:]):.2f}")

plt.figure(figsize=(8, 4))
plt.plot(td3_rewards, label='TD3')
plt.xlabel('Episode'); plt.ylabel('Return')
plt.title('TD3: Episode Returns on DroneHoverEnv')
plt.legend()
plt.tight_layout()
plt.show()